In [1]:
import os
import time
import math
import json
import warnings
import threading
import itertools
import re
import sqlite3
import hashlib
import queue

from functools import lru_cache
from collections import deque
from shapely.strtree import STRtree
from pathlib import Path
from concurrent.futures import (
    ThreadPoolExecutor,
    as_completed,
    wait,
    FIRST_COMPLETED,
)

import numpy as np
import pandas as pd
import requests

import rasterio
from rasterio.enums import MaskFlags, Resampling
from rasterio.windows import Window

import shapely
import shapely.geometry

import pystac_client
import pystac

from pyproj import Transformer
from scipy.spatial import cKDTree

import zarr
from zarr.codecs import BloscCodec, BloscShuffle

from tqdm.auto import tqdm

from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from pystac_client.stac_api_io import StacApiIO


In [2]:
# ============================================================
# CONFIGURATION
# ============================================================

PC_API_URL = "https://planetarycomputer.microsoft.com/api"
PC_STAC_URL = f"{PC_API_URL}/stac/v1"


def _normalize_pc_stac_url(value):
    """Accept a plain URL or Markdown-linked URL, but return the canonical PC endpoint."""
    text = str(value).strip()
    markdown = re.fullmatch(r"\[([^\]]+)\]\((https?://[^)]+)\)", text)
    if markdown:
        text = markdown.group(2).strip()
    text = text.strip("<> ").rstrip("/")
    expected = "https://planetarycomputer.microsoft.com/api/stac/v1"
    if text != expected:
        raise ValueError(
            "Unexpected Planetary Computer STAC URL. Expected the canonical HTTPS "
            f"endpoint {expected!r}; got {text!r}."
        )
    return expected


PC_STAC_URL = _normalize_pc_stac_url(PC_STAC_URL)
PC_SAS_URL = f"{PC_API_URL}/sas/v1/token"

S1_COLLECTION = "sentinel-1-rtc"

OUTPUT_DIR = Path(
    "Data/Global/pipeline_stage1/sentinel1"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Production aggregation method selected from the benchmark.
# "overview" is much faster but is approximate relative to exact native
# 10 m -> 1 km means. Set to "exact" to preserve the native-resolution mean.
PRODUCTION_READ_STRATEGY = "overview"  # allowed: "overview", "exact"
if PRODUCTION_READ_STRATEGY not in {"overview", "exact"}:
    raise ValueError("PRODUCTION_READ_STRATEGY must be 'overview' or 'exact'.")

# The previous VV/VH-only cache is a migration source. It is read but not
# overwritten. New, combined nearest-date results go into a new namespace.
LEGACY_STAC_CACHE_DIR = OUTPUT_DIR / "stac_cache_v4"
STAC_CACHE_DIR = OUTPUT_DIR / "stac_cache_nearest_v1"
STAC_CACHE_DIR.mkdir(parents=True, exist_ok=True)

# SQLite replaces the repeatedly rewritten global JSON scene index.
SCENE_INDEX_DB_PATH = OUTPUT_DIR / "scene_inventory_nearest_v1.sqlite"

# Keep output namespaces strategy-specific: do not mix exact values with
# overview-resampled values, even though their array shapes are identical.
if PRODUCTION_READ_STRATEGY == "overview":
    SCENE_OBS_DIR = OUTPUT_DIR / "scene_observations_nearest_overview_v1"
    FINAL_OUTPUT_DIR = OUTPUT_DIR / "nearest_valid_overview_v1"
    SCHEMA_VERSION = 6
else:
    # Preserve the old exact-read path, including its completed scene shards.
    SCENE_OBS_DIR = OUTPUT_DIR / "scene_observations_nearest_v1"
    FINAL_OUTPUT_DIR = OUTPUT_DIR / "nearest_valid_v5"
    SCHEMA_VERSION = 5

SCENE_OBS_DIR.mkdir(parents=True, exist_ok=True)
FINAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# OPTIONAL PARALLEL SCENE WORKERS (multiple local Python kernels or VMs)
# ----------------------------------------------------------------------
# Each worker gets a disjoint, deterministic subset of scene IDs and writes to
# its own subdirectory. This divides compute work; it does not create separate
# service quotas. Keep this False on the coordinator unless this is a worker.
# For a first test with TWO workers, set True in both worker copies, use IDs 0
# and 1, and set count=2 in both. Each process adjusts concurrency so the
# combined target remains near the previously tested 18 scene workers.
DISTRIBUTED_SCENE_WORKER_MODE = True
DISTRIBUTED_WORKER_ID = 0
DISTRIBUTED_WORKER_COUNT = 2
DISTRIBUTED_COMPLETED_SCENES_FILE = OUTPUT_DIR / "distributed_completed_scene_ids.txt"
DISTRIBUTED_WORKER_OUTPUT_DIR = SCENE_OBS_DIR / (
    f"_distributed_worker_{DISTRIBUTED_WORKER_ID:02d}_of_{DISTRIBUTED_WORKER_COUNT:02d}"
)
DISTRIBUTED_COMPLETED_SCENE_IDS = set()

if DISTRIBUTED_WORKER_COUNT < 1:
    raise ValueError("DISTRIBUTED_WORKER_COUNT must be >= 1")
if not 0 <= DISTRIBUTED_WORKER_ID < DISTRIBUTED_WORKER_COUNT:
    raise ValueError("DISTRIBUTED_WORKER_ID must be in 0..DISTRIBUTED_WORKER_COUNT-1")
if DISTRIBUTED_SCENE_WORKER_MODE:
    DISTRIBUTED_WORKER_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def _active_scene_shard_dir():
    """Route each worker to its own shard folder; master mode uses the normal folder."""
    return DISTRIBUTED_WORKER_OUTPUT_DIR if DISTRIBUTED_SCENE_WORKER_MODE else SCENE_OBS_DIR

# ------------------------------------------------------------
# Spatial grid
# ------------------------------------------------------------
PATCH_SIZE = 13
MODEL_RESOLUTION_M = 1000
S1_RESOLUTION_M = 10
PATCH_WIDTH_M = PATCH_SIZE * MODEL_RESOLUTION_M
PATCH_RADIUS_M = PATCH_WIDTH_M / 2
SOURCE_PATCH_PX = int(round(PATCH_WIDTH_M / S1_RESOLUTION_M))
HALF_PATCH_PX = SOURCE_PATCH_PX // 2
SOURCE_PIXELS_PER_MODEL_CELL = SOURCE_PATCH_PX // PATCH_SIZE
assert SOURCE_PATCH_PX % PATCH_SIZE == 0

# ------------------------------------------------------------
# Nearest-acquisition temporal rule
# ------------------------------------------------------------
# Select the closest acquisition with valid data, before OR after the sample
# date. The limit is a search tolerance, not a rolling/median window.
MAX_NEAREST_DAYS = 18
# Retained for the storage-aware clustering score, which uses temporal overlap.
S1_WINDOW_DAYS = MAX_NEAREST_DAYS
# A scene/polarization pair is considered usable when at least this many of
# the 169 1-km cells have a jointly valid value across the pair.
MIN_VALID_PAIR_PIXELS = 1
SOURCE_OFFSET_MISSING = int(np.iinfo(np.int16).max)

# ------------------------------------------------------------
# Polarizations
# ------------------------------------------------------------
# Keep the original four-band shard/output layout so existing v9 checkpoints
# remain readable, but only download VV and VH going forward.
REAL_BANDS = ["vv", "vh", "hh", "hv"]
DERIVED_BANDS = ["vv_vh_ratio"]
OUTPUT_BANDS = REAL_BANDS + DERIVED_BANDS
N_REAL_BANDS = len(REAL_BANDS)
N_OUTPUT_BANDS = len(OUTPUT_BANDS)
POLARIZATIONS = tuple(REAL_BANDS)  # stored-band schema, retained for compatibility
REQUIRED_POLARIZATIONS = ("vv", "vh")
READ_POLARIZATIONS = REQUIRED_POLARIZATIONS

# ------------------------------------------------------------
# Storage-aware clustering
# ------------------------------------------------------------
MAX_CLUSTER_SIDE_KM = 50.0
MAX_CLUSTER_STATIONS = 100
COG_BLOCK_PX = 512
COG_BLOCK_M = COG_BLOCK_PX * S1_RESOLUTION_M / 1000.0
MERGE_SLACK = 1.10

# ------------------------------------------------------------
# Concurrency and retries
# ------------------------------------------------------------
SEARCH_WORKERS = 3
# Historical measurements showed little throughput gain beyond 16-18 workers.
# Each scene can read VV and VH concurrently, so 18 scene workers allow up to 36
# band jobs in flight. Starting at 48 scenes would allow 96 band jobs and proved
# too aggressive in the saved run (range-read and SAS-token timeouts).
SCENE_WORKERS = 18
# Overlap network I/O for independent polarization COG assets inside each scene.
# This changes scheduling only; each asset is still read by the same v9 reader.
PARALLEL_POLARIZATION_READS = True
POLARIZATION_READ_WORKERS = 2
YEAR_WORKERS = 1
SEARCH_RETRIES = 3
STAC_REQUEST_TIMEOUT_SECONDS = 45
READ_RETRIES = 2
SCENE_RETRIES = 2
PC_TOKEN_REFRESH_MARGIN_S = 600
PC_TOKEN_CONNECT_TIMEOUT_SECONDS = 5
PC_TOKEN_READ_TIMEOUT_SECONDS = 20
PC_TOKEN_RETRIES = 3
PC_TOKEN_COOLDOWN_SECONDS = 30
SCENE_FAILURE_STREAK_LIMIT = 3

# ------------------------------------------------------------
# Latency optimisations (see the 'Latency-optimised scene runtime' cell)
# ------------------------------------------------------------
# Direct reads were tested on real data and FAILED equivalence (23/23 patch
# values differed; 5 NaN masks differed). Keep the original boundless read path.
# Do not turn this on unless a new alignment-safe implementation passes validation.
DIRECT_INSIDE_READS = False
# Threads that compress + fsync scene shards, off the network workers' path.
SHARD_WRITER_WORKERS = 4
# After a failure streak the concurrency limit halves, never below this.
MIN_SCENE_WORKERS = 4

# If multiple Python instances share one laptop, don't multiply the original
# concurrency by the number of processes. Keep aggregate target near 18 scene
# workers (each scene may read VV and VH concurrently). Example: 2 processes x
# 9 scene workers each; 3 x 6; 4 x 4. More processes may still fail to improve
# throughput if the shared network or remote service is the bottleneck.
if DISTRIBUTED_SCENE_WORKER_MODE:
    SCENE_WORKERS = max(2, 18 // DISTRIBUTED_WORKER_COUNT)
    MIN_SCENE_WORKERS = min(4, SCENE_WORKERS)

# Jobs buffered ahead of the workers = SCENE_WORKERS * this factor.
SCENE_QUEUE_FACTOR = 4
# Retry scenes that failed once more, at lower concurrency, before giving up.
SCENE_FAILED_REPASS = True
# Keep metadata discovery independent of high-latency raster reads.
STREAM_SCENES_DURING_INVENTORY = False

# ------------------------------------------------------------
# GDAL HTTP settings
# ------------------------------------------------------------
GDAL_ENV = dict(
    GDAL_DISABLE_READDIR_ON_OPEN="EMPTY_DIR",
    CPL_VSIL_CURL_ALLOWED_EXTENSIONS=".tif,.tiff,.TIF,.TIFF",
    CPL_VSIL_CURL_USE_HEAD="NO",
    # First GET returns the header AND (in a COG) the coarse overview tiles that
    # sit right behind it, so a scene-band costs one round trip. The diagnostics
    # cell can measure the exact size needed.
    GDAL_INGESTED_BYTES_AT_OPEN=str(512 * 1024),
    GDAL_HTTP_MAX_RETRY="3",
    GDAL_HTTP_RETRY_DELAY="1",
    GDAL_HTTP_RETRY_CODES="429,500,502,503,504",
    GDAL_HTTP_TIMEOUT="30",
    # Use HTTP/2 over TLS when the installed GDAL/libcurl supports it.
    GDAL_HTTP_VERSION="2TLS",
    GDAL_HTTP_CONNECTTIMEOUT="10",
    GDAL_HTTP_MERGE_CONSECUTIVE_RANGES="YES",
    GDAL_HTTP_MULTIRANGE="YES",
    GDAL_HTTP_MULTIPLEX="YES",
    VSI_CACHE="TRUE",
    VSI_CACHE_SIZE=str(8 * 1024 * 1024),
    # Global /vsicurl region cache: keep headers/tiles of many concurrent files.
    CPL_VSIL_CURL_CACHE_SIZE=str(128 * 1024 * 1024),
    GDAL_CACHEMAX=64,
)

print("Output:", OUTPUT_DIR)
print("Legacy STAC cache:", LEGACY_STAC_CACHE_DIR)
print("New STAC cache:", STAC_CACHE_DIR)
print("Scene inventory DB:", SCENE_INDEX_DB_PATH)
print("Scene observation shards:", SCENE_OBS_DIR)
print("Final output:", FINAL_OUTPUT_DIR)
print("Patch:", f"{PATCH_SIZE} x {PATCH_SIZE} @ {MODEL_RESOLUTION_M} m")
print("Source resolution:", S1_RESOLUTION_M, "m")
print("Nearest-date tolerance:", MAX_NEAREST_DAYS, "days")
print("Stored bands:", OUTPUT_BANDS)
print("Remote polarizations to read:", READ_POLARIZATIONS)
print("Production raster read strategy:", PRODUCTION_READ_STRATEGY)
if PRODUCTION_READ_STRATEGY == "overview":
    print("Overview resampling:", "average; approximate versus exact native-pixel means")
print("STAC workers:", SEARCH_WORKERS)
print("GDAL HTTP version preference:", GDAL_ENV["GDAL_HTTP_VERSION"])
print("GDAL per-request timeout:", GDAL_ENV["GDAL_HTTP_TIMEOUT"], "seconds")
print("Streaming scenes during inventory:", STREAM_SCENES_DURING_INVENTORY)
print("Scene workers:", SCENE_WORKERS)
print("Direct inside-raster reads:", DIRECT_INSIDE_READS, "| shard writer threads:", SHARD_WRITER_WORKERS)
print("Parallel polarization reads:", PARALLEL_POLARIZATION_READS)
print("Max simultaneous polarization assets per scene:", POLARIZATION_READ_WORKERS)
print("COG tile model:", f"{COG_BLOCK_PX} x {COG_BLOCK_PX} ({COG_BLOCK_M:.2f} km)")


Output: Data/Global/pipeline_stage1/sentinel1
Legacy STAC cache: Data/Global/pipeline_stage1/sentinel1/stac_cache_v4
New STAC cache: Data/Global/pipeline_stage1/sentinel1/stac_cache_nearest_v1
Scene inventory DB: Data/Global/pipeline_stage1/sentinel1/scene_inventory_nearest_v1.sqlite
Scene observation shards: Data/Global/pipeline_stage1/sentinel1/scene_observations_nearest_overview_v1
Final output: Data/Global/pipeline_stage1/sentinel1/nearest_valid_overview_v1
Patch: 13 x 13 @ 1000 m
Source resolution: 10 m
Nearest-date tolerance: 18 days
Stored bands: ['vv', 'vh', 'hh', 'hv', 'vv_vh_ratio']
Remote polarizations to read: ('vv', 'vh')
Production raster read strategy: overview
Overview resampling: average; approximate versus exact native-pixel means
STAC workers: 3
GDAL HTTP version preference: 2TLS
GDAL per-request timeout: 30 seconds
Streaming scenes during inventory: False
Scene workers: 9
Direct inside-raster reads: False | shard writer threads: 4
Parallel polarization reads: True
M

In [3]:
# ============================================================
# CONFIGURATION SANITY CHECKS
# ============================================================

assert SCENE_OBS_DIR.exists()
assert SCENE_INDEX_DB_PATH.parent.exists()
assert FINAL_OUTPUT_DIR.exists()
assert REQUIRED_POLARIZATIONS == ("vv", "vh")
assert READ_POLARIZATIONS == ("vv", "vh")
assert all(pol in REAL_BANDS for pol in REQUIRED_POLARIZATIONS)
assert SCENE_WORKERS >= 1
assert SEARCH_WORKERS >= 1
assert POLARIZATION_READ_WORKERS >= 1
assert isinstance(GDAL_ENV.get("GDAL_CACHEMAX"), int)
assert STAC_REQUEST_TIMEOUT_SECONDS > 0
assert MAX_NEAREST_DAYS >= 0
assert len(OUTPUT_BANDS) == 5

print("Configuration OK")
print("Scene shards:", SCENE_OBS_DIR)
print("Scene inventory DB:", SCENE_INDEX_DB_PATH)
print("STAC search workers:", SEARCH_WORKERS)
print("Scene read workers:", SCENE_WORKERS)
print("GDAL cache max:", GDAL_ENV["GDAL_CACHEMAX"], "MB")
print("STAC request timeout:", STAC_REQUEST_TIMEOUT_SECONDS, "seconds")
print("Nearest-date tolerance:", MAX_NEAREST_DAYS, "days")


Configuration OK
Scene shards: Data/Global/pipeline_stage1/sentinel1/scene_observations_nearest_overview_v1
Scene inventory DB: Data/Global/pipeline_stage1/sentinel1/scene_inventory_nearest_v1.sqlite
STAC search workers: 3
Scene read workers: 9
GDAL cache max: 64 MB
STAC request timeout: 45 seconds
Nearest-date tolerance: 18 days


### Production settings

The scene inventory is stored incrementally in SQLite, not by repeatedly rewriting one giant JSON index. For each spatial cluster, STAC searches are limited to the **union of the date intervals actually needed by the samples** (sample date ± `MAX_NEAREST_DAYS`), rather than querying every day between the cluster's first and last sample date. Overlapping intervals are merged, the query result for each interval/polarization pair is cached independently, and completed cluster results are checkpointed. Newly discovered scenes enter a bounded remote-COG worker queue while inventory continues; global station-patch indexing ensures each scene shard includes relevant stations regardless of which cluster discovered the scene first. This lets an interrupted run resume without repeating successful requests.

For each station/sample date, output uses the nearest acquisition with valid data on either side of the target date; an exact distance tie prefers the earlier acquisition. There is no temporal median. The current safety tolerance is configurable as `MAX_NEAREST_DAYS` (18 days by default); no observation inside that tolerance leaves the sample as NaN. Chosen acquisition offsets are stored in the Zarr output.


**Fast VV/VH-only mode:** the production scene reader downloads only VV and VH. HH/HV are excluded before any remote raster open. The original four-band checkpoint/output layout is retained so existing v9 scene shards stay compatible; pre-existing cached HH/HV values are not deleted, while newly read scene shards leave those two slots empty.


## Planetary Computer metadata + remote COG range processing

The STAC metadata API is public. Azure raster assets are read using a Planetary Computer SAS token; no subscription/API key is required by this notebook.

The data path is:

1. For each cluster, merge the date windows surrounding actual sample dates; do not query the empty gaps between widely separated target dates.
2. Cache each STAC interval/polarization query and commit completed cluster results incrementally to SQLite.
3. Start bounded remote COG reads while inventory searches continue; a global station-patch spatial index keeps shared-scene extraction complete.
4. Read only native, tile-aligned 10 m COG windows via HTTP range requests.
5. Aggregate locally to 1 km and select the single nearest valid acquisition for each sample/polarization pair (earlier date wins ties).
6. Store scene observations and final yearly Zarr arrays so interrupted runs can resume.

`MAX_NEAREST_DAYS` defines the maximum allowed distance from a target date, not a rolling average/median window. Increase it if a wider nearest-date search is intended.


### HTTP behavior and recovery

STAC metadata searches use GET for the small bbox/date queries, with bounded transport retries and application-level backoff. Raster COG reads request HTTP/2 over TLS when the installed GDAL/libcurl supports it; GDAL's HTTP/2 multiplexing can improve parallel byte-range reads. If the local GDAL/libcurl build reports HTTP-version/protocol errors, change `GDAL_HTTP_VERSION` to `"1.1"`. HTTP/2 does not fix a machine/network outage; network-unreachable errors are retried with backoff.


In [4]:
def _clean_href(href):
    """Normalize STAC asset hrefs before appending a fresh SAS token."""
    if href is None:
        raise ValueError("Asset href is None")
    href = str(href).strip()
    if href.startswith("[") and "](" in href and href.endswith(")"):
        href = href[1:href.index("](")]
    href = href.split("?", 1)[0]
    return href


class PCTokenManager:
    """Thread-safe, auto-refreshing, anonymous SAS token for one PC collection."""

    def __init__(self, collection, sas_url=PC_SAS_URL, margin_s=PC_TOKEN_REFRESH_MARGIN_S):
        self.collection = collection
        self.sas_url = sas_url.rstrip("/")
        self.margin_s = margin_s
        self._lock = threading.Lock()
        self._session = requests.Session()
        self._token = None
        self.expiry = None
        self.lifetime_s = None
        self._retry_after_monotonic = 0.0
        self._last_error = None

    def seconds_left(self) -> float:
        if self.expiry is None:
            return -1.0
        return (self.expiry - pd.Timestamp.now(tz="UTC")).total_seconds()

    def _margin(self) -> float:
        # Never let the margin exceed a quarter of the token lifetime, otherwise a
        # short-lived token would be re-minted on every single call.
        if self.lifetime_s:
            return min(self.margin_s, 0.25 * self.lifetime_s)
        return self.margin_s

    @staticmethod
    def _retry_delay(resp, attempt) -> float:
        try:
            return min(float(resp.headers["Retry-After"]), 30.0)
        except (KeyError, ValueError):
            return float(min(2 ** attempt, 30))

    def _fetch(self):
        now = time.monotonic()
        if now < self._retry_after_monotonic:
            raise requests.ConnectionError(
                "Planetary Computer token endpoint is in a short cooldown after a recent failure: "
                f"{self._last_error or 'connection failure'}"
            )

        url = f"{self.sas_url}/{self.collection}"
        last_error = None
        for attempt in range(PC_TOKEN_RETRIES):
            resp = None
            try:
                resp = self._session.get(
                    url,
                    timeout=(PC_TOKEN_CONNECT_TIMEOUT_SECONDS, PC_TOKEN_READ_TIMEOUT_SECONDS),
                )
                if resp.status_code in (429, 500, 502, 503, 504) and attempt < PC_TOKEN_RETRIES - 1:
                    delay = self._retry_delay(resp, attempt)
                    resp.close()
                    time.sleep(min(delay, 4.0))
                    continue
                resp.raise_for_status()
                body = resp.json()
                expiry = pd.Timestamp(body["msft:expiry"])
                self.expiry = (
                    expiry.tz_localize("UTC") if expiry.tzinfo is None
                    else expiry.tz_convert("UTC")
                )
                self._token = body["token"]
                self.lifetime_s = self.seconds_left()
                self._retry_after_monotonic = 0.0
                self._last_error = None
                return
            except requests.RequestException as exc:
                last_error = exc
                if resp is not None:
                    resp.close()
                if attempt >= PC_TOKEN_RETRIES - 1:
                    self._last_error = f"{type(exc).__name__}: {exc}"
                    self._retry_after_monotonic = time.monotonic() + PC_TOKEN_COOLDOWN_SECONDS
                    raise
                delay = min(4.0, 2 ** attempt)
                print(
                    f"Planetary Computer token request failed "
                    f"(attempt {attempt + 1}/{PC_TOKEN_RETRIES}): "
                    f"{type(exc).__name__}: {exc}; retrying in {delay:.0f}s."
                )
                time.sleep(delay)
            except (KeyError, ValueError, TypeError) as exc:
                if resp is not None:
                    resp.close()
                self._last_error = f"invalid token response: {exc}"
                self._retry_after_monotonic = time.monotonic() + PC_TOKEN_COOLDOWN_SECONDS
                raise RuntimeError(f"Invalid Planetary Computer SAS token response: {exc}") from exc
        if last_error is not None:
            self._last_error = f"{type(last_error).__name__}: {last_error}"
            self._retry_after_monotonic = time.monotonic() + PC_TOKEN_COOLDOWN_SECONDS
            raise last_error
        raise requests.ConnectionError("Planetary Computer SAS token request did not complete")

    def token(self, force: bool = False) -> str:
        with self._lock:
            if force or self._token is None or self.seconds_left() < self._margin():
                self._fetch()
            return self._token

    def sign(
            self,
            href: str,
            force: bool = False,
        ) -> str:

            href = _clean_href(href)

            return (
                f"{href}?{self.token(force)}"
            )


rtc_token = PCTokenManager(S1_COLLECTION)
print("SAS token manager configured; token is fetched on demand during scene extraction.")


SAS token manager configured; token is fetched on demand during scene extraction.


In [5]:
if "samples" not in globals():
    samples = pd.read_parquet(
        "Data/Global/pipeline_stage1/station_date_candidates.parquet"
    )

if "stations" not in globals():
    stations = pd.read_parquet(
        "Data/Global/pipeline_stage1/stations.parquet"
    )

samples = samples.copy()
stations = stations.copy()

for name, df, cols in [
    ("samples", samples, {"station_id", "sample_date"}),
    ("stations", stations, {"station_id", "latitude", "longitude"}),
]:
    missing = cols - set(df.columns)
    if missing:
        raise KeyError(f"{name} is missing columns: {sorted(missing)}")

samples["sample_date"] = pd.to_datetime(samples["sample_date"]).dt.normalize()
samples["station_id"] = samples["station_id"].astype(str)
stations["station_id"] = stations["station_id"].astype(str)

# ---- hygiene: bad coordinates, duplicates, stations nobody needs ----
n_st0, n_sm0 = len(stations), len(samples)

stations = stations.dropna(subset=["latitude", "longitude"])
stations = stations[
    stations["latitude"].between(-90, 90)
    & stations["longitude"].between(-180, 180)
]
stations = stations.drop_duplicates("station_id").reset_index(drop=True)

samples = samples.dropna(subset=["sample_date"])
samples = samples[samples["station_id"].isin(stations["station_id"])]

# A station with no samples must never be downloaded.
stations = stations[stations["station_id"].isin(samples["station_id"])]
stations = stations.reset_index(drop=True)
samples = samples.reset_index(drop=True)

print(f"Stations: {len(stations):,}  (dropped {n_st0 - len(stations):,})")
print(f"Samples:  {len(samples):,}  (dropped {n_sm0 - len(samples):,})")
print("Date range:", samples["sample_date"].min(), "->", samples["sample_date"].max())

Stations: 2,210  (dropped 0)
Samples:  3,514,643  (dropped 0)
Date range: 2015-04-01 00:00:00 -> 2026-10-03 00:00:00


In [6]:
# STAC search is public metadata; production workers create their own client
# per thread so one HTTP client is never shared concurrently across workers.
catalog = pystac_client.Client.open(_normalize_pc_stac_url(PC_STAC_URL))
_ = catalog.get_collection(S1_COLLECTION)

print("Planetary Computer STAC connected:", S1_COLLECTION)


Planetary Computer STAC connected: sentinel-1-rtc


In [7]:
# ============================================================
# GEOMETRY + SAMPLE-DATE INDEX
# ============================================================

EARTH_R_KM = 6371.0088
EPOCH = pd.Timestamp("1970-01-01")


def ground_xy_km(
    lon,
    lat,
):
    """
    Approximate local metric coordinates in km.

    Used only for the storage-aware station clustering.
    Actual raster geometry is always handled in the scene's
    own projected CRS.
    """

    lat_r = np.radians(lat)

    return (
        EARTH_R_KM
        * np.radians(lon)
        * np.cos(lat_r),

        EARTH_R_KM
        * lat_r,
    )


def station_patch_boxes_deg(
    df,
):
    """
    Approximate 13 km x 13 km station footprints in lon/lat.

    Used for STAC footprint filtering only.

    Actual Sentinel-1 reads are constructed in the scene's
    projected CRS from exact pixel geometry.
    """

    lon = df[
        "longitude"
    ].to_numpy(float)

    lat = df[
        "latitude"
    ].to_numpy(float)

    radius_km = (
        PATCH_RADIUS_M / 1000.0
    )

    dlat = np.degrees(
        radius_km / EARTH_R_KM
    )

    lat_edge = np.minimum(
        np.abs(lat) + dlat,
        89.0,
    )

    dlon = np.degrees(
        radius_km
        /
        (
            EARTH_R_KM
            *
            np.cos(
                np.radians(
                    lat_edge
                )
            )
        )
    )

    return np.column_stack(
        [
            np.clip(
                lon - dlon,
                -180,
                180,
            ),

            np.clip(
                lat - dlat,
                -90,
                90,
            ),

            np.clip(
                lon + dlon,
                -180,
                180,
            ),

            np.clip(
                lat + dlat,
                -90,
                90,
            ),
        ]
    )


def sample_days_by_station(
    samples_df,
):
    """
    Sorted unique sample days per station.
    """

    result = {}

    for (
        station_id,
        group,
    ) in samples_df.groupby(
        "station_id"
    ):

        days = (
            group[
                "sample_date"
            ]
            .to_numpy(
                dtype="datetime64[D]"
            )
            .astype(
                np.int64
            )
        )

        result[
            station_id
        ] = np.unique(
            days
        )

    return result


# Full global sample-date index.
SAMPLE_DAYS = (
    sample_days_by_station(
        samples
    )
)


print(
    "COG tile model:",
    f"{COG_BLOCK_PX} x {COG_BLOCK_PX}",
)

print(
    "Native patch:",
    f"{SOURCE_PATCH_PX} x {SOURCE_PATCH_PX}",
)

print(
    "Native pixels / model cell:",
    SOURCE_PIXELS_PER_MODEL_CELL,
)

COG tile model: 512 x 512
Native patch: 1300 x 1300
Native pixels / model cell: 100


## Clustering

Recursive bisection in ground-km space with a storage-aware cut rule.

The weighted graph favors keeping stations together when their 13 km patches are likely to share 512×512 RTC COG tiles and their sample dates overlap within the nearest-acquisition tolerance. The hard cluster diameter/station limits cap memory and prevent long chain-shaped clusters.


In [8]:
# ============================================================
# STORAGE-AWARE CLUSTERING
# ============================================================

def temporal_overlap_score(
    days_a,
    days_b,
    window_days=S1_WINDOW_DAYS,
):
    """
    Approximate number of source windows that can share
    Sentinel-1 acquisitions.

    We don't count every possible date pair. For every sample
    day at A, we only ask whether B has at least one sample
    within +/- window_days, and vice versa.
    """

    if (
        len(days_a) == 0
        or len(days_b) == 0
    ):
        return 0.0

    a = np.asarray(
        days_a,
        dtype=np.int64,
    )

    b = np.asarray(
        days_b,
        dtype=np.int64,
    )

    def one_way(
        x,
        y,
    ):
        k = np.searchsorted(
            y,
            x,
            side="left",
        )

        k0 = np.clip(
            k - 1,
            0,
            len(y) - 1,
        )

        k1 = np.clip(
            k,
            0,
            len(y) - 1,
        )

        distance = np.minimum(
            np.abs(
                x - y[k0]
            ),
            np.abs(
                x - y[k1]
            ),
        )

        return (
            distance
            <= window_days
        )

    return 0.5 * (
        one_way(a, b).sum()
        +
        one_way(b, a).sum()
    )


def tile_box_for_station(
    x,
    y,
    phase_x,
    phase_y,
):
    """
    Approximate source COG tiles touched by one station patch.

    We average across several unknown tile-grid phases because
    each RTC scene's projected grid origin can differ.
    """

    block = COG_BLOCK_M

    r = PATCH_RADIUS_M / 1000

    return np.array(
        [
            math.floor(
                (x - r - phase_x)
                / block
            ),

            math.floor(
                (y - r - phase_y)
                / block
            ),

            math.ceil(
                (x + r - phase_x)
                / block
            ),

            math.ceil(
                (y + r - phase_y)
                / block
            ),
        ],
        dtype=np.int64,
    )


def expected_shared_tile_savings(
    x1,
    y1,
    x2,
    y2,
):
    """
    Expected number of internal COG tiles saved when two
    station windows are read together rather than separately.
    """

    phases = [
        0.00,
        0.25 * COG_BLOCK_M,
        0.50 * COG_BLOCK_M,
        0.75 * COG_BLOCK_M,
    ]

    values = []

    for phase_x in phases:

        for phase_y in phases:

            a = tile_box_for_station(
                x1,
                y1,
                phase_x,
                phase_y,
            )

            b = tile_box_for_station(
                x2,
                y2,
                phase_x,
                phase_y,
            )

            area_a = (
                a[2] - a[0]
            ) * (
                a[3] - a[1]
            )

            area_b = (
                b[2] - b[0]
            ) * (
                b[3] - b[1]
            )

            union = (
                max(
                    a[2],
                    b[2],
                )
                -
                min(
                    a[0],
                    b[0],
                )
            ) * (
                max(
                    a[3],
                    b[3],
                )
                -
                min(
                    a[1],
                    b[1],
                )
            )

            values.append(
                max(
                    0,
                    area_a
                    + area_b
                    - union,
                )
            )

    return float(
        np.mean(values)
    )


def build_storage_edges(
    stations_df,
):
    """
    Build a weighted graph.

    Edge weight approximates:

        shared COG tiles
        ×
        temporal co-occurrence

    Only geographically plausible pairs are considered.
    """

    x, y = ground_xy_km(
        stations_df[
            "longitude"
        ].to_numpy(float),

        stations_df[
            "latitude"
        ].to_numpy(float),
    )

    coords = np.column_stack(
        [x, y]
    )

    share_distance_km = (
        2
        * PATCH_RADIUS_M
        / 1000
        + COG_BLOCK_M
    )

    pairs = (
        cKDTree(
            coords
        )
        .query_pairs(
            share_distance_km,
            p=np.inf,
            output_type="ndarray",
        )
    )

    station_ids = (
        stations_df[
            "station_id"
        ].to_numpy()
    )

    edges = []

    for i, j in pairs:

        spatial = (
            expected_shared_tile_savings(
                x[i],
                y[i],
                x[j],
                y[j],
            )
        )

        if spatial <= 0:
            continue

        temporal = (
            temporal_overlap_score(
                SAMPLE_DAYS[
                    station_ids[i]
                ],

                SAMPLE_DAYS[
                    station_ids[j]
                ],
            )
        )

        if temporal <= 0:
            continue

        edges.append(
            (
                int(i),
                int(j),
                spatial * temporal,
            )
        )

    if not edges:
        return np.empty(
            (0, 3),
            dtype=np.float64,
        )

    return np.asarray(
        edges,
        dtype=np.float64,
    )


def build_station_clusters(
    stations_df,
):
    """
    Recursive bisection.

    A cut is preferred when it severs the smallest amount of
    expected Azure tile reuse.

    Only the hard side/station limits force a split.
    """

    x, y = ground_xy_km(
        stations_df[
            "longitude"
        ].to_numpy(float),

        stations_df[
            "latitude"
        ].to_numpy(float),
    )

    n = len(
        stations_df
    )

    edges = build_storage_edges(
        stations_df
    )

    labels = np.full(
        n,
        -1,
        dtype=np.int64,
    )

    stack = [
        np.arange(
            n,
            dtype=np.int64,
        )
    ]

    next_cluster = 0

    while stack:

        idx = stack.pop()

        width = np.ptp(
            x[idx]
        )

        height = np.ptp(
            y[idx]
        )

        if (
            len(idx)
            <= MAX_CLUSTER_STATIONS
            and
            width
            <= MAX_CLUSTER_SIDE_KM
            and
            height
            <= MAX_CLUSTER_SIDE_KM
        ):

            labels[idx] = (
                next_cluster
            )

            next_cluster += 1
            continue

        axis = (
            x
            if width >= height
            else y
        )

        ordered = idx[
            np.argsort(
                axis[idx],
                kind="stable",
            )
        ]

        values = axis[
            ordered
        ]

        m = len(
            ordered
        )

        lo = max(
            1,
            m // 4,
        )

        hi = min(
            m - 1,
            m - m // 4,
        )

        gaps = np.diff(
            values
        )[
            lo - 1:hi
        ]

        # Global station -> local order.
        position = np.full(
            n,
            -1,
            dtype=np.int64,
        )

        position[
            ordered
        ] = np.arange(
            m
        )

        if len(edges):

            e0 = edges[
                :, 0
            ].astype(
                np.int64
            )

            e1 = edges[
                :, 1
            ].astype(
                np.int64
            )

            inside = (
                (position[e0] >= 0)
                &
                (position[e1] >= 0)
            )

            a = position[
                e0[inside]
            ]

            b = position[
                e1[inside]
            ]

            weights = edges[
                inside,
                2,
            ]

            left = np.minimum(
                a,
                b,
            )

            right = np.maximum(
                a,
                b,
            )

            diff = np.zeros(
                m + 1,
                dtype=np.float64,
            )

            np.add.at(
                diff,
                left + 1,
                weights,
            )

            np.add.at(
                diff,
                right + 1,
                -weights,
            )

            severed = np.cumsum(
                diff
            )[
                lo:hi + 1
            ]

        else:

            severed = np.zeros(
                hi - lo + 1,
                dtype=np.float64,
            )

        best = np.flatnonzero(
            severed
            == severed.min()
        )

        # After minimizing Azure tile reuse severed,
        # use the widest geographic gap as a tie-break.
        if (
            gaps[
                best
            ].max()
            > 0
        ):

            k = (
                lo
                +
                int(
                    best[
                        np.argmax(
                            gaps[
                                best
                            ]
                        )
                    ]
                )
            )

        else:

            k = (
                lo
                +
                int(
                    best[
                        np.argmin(
                            np.abs(
                                lo
                                + best
                                - m / 2
                            )
                        )
                    ]
                )
            )

        stack.append(
            ordered[:k]
        )

        stack.append(
            ordered[k:]
        )

    return pd.DataFrame(
        {
            "station_id":
                stations_df[
                    "station_id"
                ].to_numpy(),

            "cluster_id":
                labels,
        }
    )


station_clusters = (
    build_station_clusters(
        stations
    )
)

stations_cluster = (
    stations
    .merge(
        station_clusters,
        on="station_id",
        how="left",
        validate="one_to_one",
    )
)

assert (
    stations_cluster[
        "cluster_id"
    ]
    >= 0
).all()

# Global spatial index for safe inventory/extraction overlap. It is built once
# and avoids scanning every station for every scene. STRtree query results are
# station-row positions in GLOBAL_STATION_FRAME.
GLOBAL_STATION_FRAME = stations_cluster[
    ["station_id", "latitude", "longitude", "cluster_id"]
].copy().reset_index(drop=True)
_GLOBAL_PATCH_BOXES = station_patch_boxes_deg(GLOBAL_STATION_FRAME)
GLOBAL_STATION_PATCH_GEOMETRIES = shapely.box(
    _GLOBAL_PATCH_BOXES[:, 0],
    _GLOBAL_PATCH_BOXES[:, 1],
    _GLOBAL_PATCH_BOXES[:, 2],
    _GLOBAL_PATCH_BOXES[:, 3],
)
GLOBAL_STATION_TREE = STRtree(GLOBAL_STATION_PATCH_GEOMETRIES)
print("Global station-patch spatial index:", len(GLOBAL_STATION_FRAME), "stations")

print(
    "Clusters:",
    stations_cluster[
        "cluster_id"
    ].nunique(),
)

print(
    "Largest cluster:",
    stations_cluster.groupby(
        "cluster_id"
    ).size().max(),
)

Global station-patch spatial index: 2210 stations
Clusters: 819
Largest cluster: 50


In [9]:
# ============================================================
# CLUSTER TABLE
# ============================================================

def make_cluster_table(
    stations_df,
    samples_df,
):
    s = stations_df[
        [
            "station_id",
            "cluster_id",
            "latitude",
            "longitude",
        ]
    ].copy()

    boxes = (
        station_patch_boxes_deg(
            s
        )
    )

    s[
        "b0"
    ], s[
        "b1"
    ], s[
        "b2"
    ], s[
        "b3"
    ] = boxes.T

    x, y = ground_xy_km(
        s["longitude"].to_numpy(float),
        s["latitude"].to_numpy(float),
    )

    s["x"] = x
    s["y"] = y

    g = (
        s.groupby(
            "cluster_id"
        )
        .agg(
            bbox_min_lon=("b0", "min"),
            bbox_min_lat=("b1", "min"),
            bbox_max_lon=("b2", "max"),
            bbox_max_lat=("b3", "max"),

            n_stations=(
                "station_id",
                "size",
            ),

            x0=("x", "min"),
            x1=("x", "max"),
            y0=("y", "min"),
            y1=("y", "max"),
        )
    )

    g[
        "width_km"
    ] = (
        g["x1"]
        - g["x0"]
        + PATCH_WIDTH_M / 1000
    )

    g[
        "height_km"
    ] = (
        g["y1"]
        - g["y0"]
        + PATCH_WIDTH_M / 1000
    )

    d = (
        samples_df[
            [
                "station_id",
                "sample_date",
            ]
        ]

        .merge(
            s[
                [
                    "station_id",
                    "cluster_id",
                ]
            ],

            on="station_id",
        )

        .groupby(
            "cluster_id"
        )["sample_date"]

        .agg(
            min_date="min",
            max_date="max",
            n_samples="size",
        )
    )

    return (
        g
        .drop(
            columns=[
                "x0",
                "x1",
                "y0",
                "y1",
            ]
        )

        .join(d)

        .reset_index()
    )


clusters = (
    make_cluster_table(
        stations_cluster,
        samples,
    )
    .sort_values(
        "cluster_id"
    )
    .reset_index(
        drop=True
    )
)

samples_c = (
    samples[
        [
            c
            for c in [
                "sample_id",
                "station_id",
                "sample_date",
            ]
            if c in samples.columns
        ]
    ]

    .merge(
        stations_cluster[
            [
                "station_id",
                "cluster_id",
            ]
        ],

        on="station_id",

        validate="many_to_one",
    )
)

samples_c["year"] = (
    samples_c[
        "sample_date"
    ].dt.year
)

clusters.to_parquet(
    OUTPUT_DIR / "clusters.parquet",
    index=False,
)

stations_cluster.to_parquet(
    OUTPUT_DIR
    / "stations_with_clusters.parquet",
    index=False,
)

print(
    "Clusters:",
    len(clusters),
)

display(
    clusters.head(20)
)

Clusters: 819


,cluster_id,bbox_min_lon,bbox_min_lat,bbox_max_lon,bbox_max_lat,n_stations,width_km,height_km,min_date,max_date,n_samples
0,0,127.588640,36.981764,127.736040,37.098906,18,13.049375,13.025575,2015-04-01,2019-12-10,22502
1,1,126.893367,37.231044,127.061034,37.352556,10,14.660379,13.511497,2015-08-31,2017-11-29,2406
2,2,126.876883,37.878624,127.025847,37.996146,19,13.119585,13.067829,2015-04-04,2019-12-04,20509
3,3,116.359200,41.515974,116.515620,41.632886,1,13.000000,13.000000,2018-09-12,2019-11-12,302
4,4,115.460668,41.492304,116.439707,42.013766,5,41.251026,57.983970,2018-09-08,2019-11-12,1487
5,5,115.602080,41.704334,116.255364,42.083556,12,18.340978,42.167581,2018-07-28,2022-11-24,5148
6,6,115.730577,42.046824,116.447015,42.459666,8,42.117303,45.905960,2018-07-25,2019-11-22,2457
7,7,115.524679,41.721614,116.264362,42.154416,12,50.882304,48.125414,2018-07-25,2019-11-22,3588
8,8,115.610656,42.199154,116.301959,42.478196,6,57.222177,31.028058,2018-08-20,2019-11-22,1673
9,9,102.842710,38.570704,102.993619,38.688426,10,13.116862,13.090068,2021-01-22,2022-12-21,6039


In [10]:
# ============================================================
# GLOBAL UNIQUE SCENE INVENTORY -- SQLITE, INCREMENTAL, RESUMABLE
# ============================================================


def cluster_cache_path(cluster_id):
    return STAC_CACHE_DIR / f"cluster_{int(cluster_id):05d}.json"


def legacy_cluster_cache_path(cluster_id):
    return LEGACY_STAC_CACHE_DIR / f"cluster_{int(cluster_id):05d}.json"


def stac_day(timestamp):
    return pd.Timestamp(timestamp).strftime("%Y-%m-%dT00:00:00Z")


def load_stac_cache(cache_path):
    if not cache_path.exists() or cache_path.stat().st_size == 0:
        return None
    try:
        with cache_path.open("r", encoding="utf-8") as f:
            raw = json.load(f)
        if not isinstance(raw, list):
            return None
        items = []
        for entry in raw:
            if not isinstance(entry, dict) or not entry.get("id"):
                return None
            items.append(pystac.Item.from_dict(entry))
        return items
    except (json.JSONDecodeError, ValueError, KeyError, TypeError, OSError):
        return None


def save_stac_cache(cache_path, items):
    """Atomic cluster cache write; interrupted runs cannot leave valid-looking partial JSON."""
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = Path(str(cache_path) + ".tmp")
    with tmp_path.open("w", encoding="utf-8") as f:
        f.write("[")
        for i, item in enumerate(items):
            if i:
                f.write(",")
            json.dump(item.to_dict(), f, separators=(",", ":"))
        f.write("]")
        f.flush()
        os.fsync(f.fileno())
    os.replace(tmp_path, cache_path)


_STAC_TLS = threading.local()
_STAC_PROGRESS_EVENTS = None


def _emit_inventory_event(kind, **payload):
    """Send lightweight worker events to the main thread for live progress."""
    event_queue = globals().get("_STAC_PROGRESS_EVENTS")
    if event_queue is not None:
        event = {"kind": kind, "at": time.monotonic()}
        event.update(payload)
        event_queue.put(event)


def _get_worker_catalog():
    """Independent STAC client with bounded transport retries per search thread."""
    client = getattr(_STAC_TLS, "catalog", None)
    if client is None:
        retry = Retry(
            total=2,
            connect=2,
            read=2,
            status=2,
            backoff_factor=0.5,
            status_forcelist=(429, 500, 502, 503, 504),
            allowed_methods=None,
            respect_retry_after_header=True,
            raise_on_status=False,
        )
        stac_io = StacApiIO()
        adapter = HTTPAdapter(
            max_retries=retry,
            pool_connections=6,
            pool_maxsize=6,
        )
        stac_io.session.mount("https://", adapter)
        stac_io.session.mount("http://", adapter)
        client = pystac_client.Client.open(
            _normalize_pc_stac_url(PC_STAC_URL),
            stac_io=stac_io,
            timeout=STAC_REQUEST_TIMEOUT_SECONDS,
        )
        _STAC_TLS.catalog = client
    return client


def _query_stac_window(bbox, start, end, cluster_id):
    """Retrieve one bounded date window with live item-count events.

    Polarizations are filtered locally after retrieval. One spatial/temporal
    request per window avoids duplicate VV/VH and HH/HV requests and also avoids
    relying on exact array equality for the STAC polarization property.
    """
    datetime_range = f"{stac_day(start)}/{stac_day(end)}"
    query_id = (
        f"{int(cluster_id)}:{datetime_range}:{threading.get_ident()}:{time.monotonic_ns()}"
    )
    _emit_inventory_event(
        "query_start",
        query_id=query_id,
        cluster_id=int(cluster_id),
        start=str(pd.Timestamp(start).date()),
        end=str(pd.Timestamp(end).date()),
        datetime_range=datetime_range,
    )

    items = []
    try:
        search = _get_worker_catalog().search(
            collections=[S1_COLLECTION],
            bbox=bbox,
            datetime=datetime_range,
            method="GET",
            limit=500,
            max_items=None,
            # Item bbox remains available for conservative local spatial filtering.
            fields={"exclude": ["geometry"]},
        )
        for count, item in enumerate(search.items(), start=1):
            items.append(item)
            # Reduce cross-thread event traffic while still showing pages flowing.
            if count == 1 or count % 25 == 0:
                _emit_inventory_event(
                    "query_items",
                    query_id=query_id,
                    cluster_id=int(cluster_id),
                    items_seen=count,
                )
        _emit_inventory_event(
            "query_end",
            query_id=query_id,
            cluster_id=int(cluster_id),
            success=True,
            items=len(items),
        )
        return items
    except Exception as exc:
        _emit_inventory_event(
            "query_end",
            query_id=query_id,
            cluster_id=int(cluster_id),
            success=False,
            items=len(items),
            error=f"{type(exc).__name__}: {exc}",
        )
        raise


def _query_window_with_retry(bbox, start, end, cluster_id):
    """Retry bounded STAC windows with exponential backoff and visible status."""
    for attempt in range(SEARCH_RETRIES):
        try:
            return _query_stac_window(bbox, start, end, cluster_id)
        except KeyboardInterrupt:
            raise
        except Exception as exc:
            if attempt >= SEARCH_RETRIES - 1:
                raise
            delay = min(60, 2 ** (attempt + 1))
            _emit_inventory_event(
                "query_retry",
                cluster_id=int(cluster_id),
                attempt=attempt + 1,
                attempts=SEARCH_RETRIES,
                delay=delay,
                error=f"{type(exc).__name__}: {exc}",
            )
            tqdm.write(
                f"STAC window failed for cluster {cluster_id} "
                f"(attempt {attempt + 1}/{SEARCH_RETRIES}): "
                f"{type(exc).__name__}: {exc}; retrying in {delay}s."
            )
            time.sleep(delay)
    return []


def _cluster_sample_day_union(cluster_stations):
    """Union of sample dates for a cluster, built once per cluster search."""
    arrays = [
        SAMPLE_DAYS.get(str(station_id))
        for station_id in cluster_stations["station_id"].astype(str).to_numpy()
    ]
    arrays = [a for a in arrays if a is not None and len(a)]
    if not arrays:
        return np.empty(0, dtype=np.int64)
    if len(arrays) == 1:
        return arrays[0]
    return np.unique(np.concatenate(arrays)).astype(np.int64, copy=False)


def _item_near_any_sample_day(item, cluster_days):
    """Fast temporal relevance test against the cluster-wide sorted date union."""
    if cluster_days is None or len(cluster_days) == 0:
        return False
    scene_day = item_day(item)
    k = int(np.searchsorted(cluster_days, scene_day, side="left"))
    nearest = np.inf
    if k < len(cluster_days):
        nearest = min(nearest, abs(int(cluster_days[k]) - scene_day))
    if k > 0:
        nearest = min(nearest, abs(int(cluster_days[k - 1]) - scene_day))
    return nearest <= MAX_NEAREST_DAYS


def _filter_cluster_items(items, cluster_row):
    """Keep relevant scenes using cached cluster dates and prebuilt station boxes."""
    cluster_id = int(cluster_row.cluster_id)
    cluster_stations = stations_cluster[
        stations_cluster["cluster_id"] == cluster_id
    ].reset_index(drop=True)
    if cluster_stations.empty:
        return []

    # This is a conservative filter: a date used by any station can admit a scene,
    # but the downstream per-station filter still enforces the exact sample window.
    cluster_days = _cluster_sample_day_union(cluster_stations)
    station_boxes = station_patch_boxes_deg(cluster_stations)
    station_geometries = shapely.box(
        station_boxes[:, 0],
        station_boxes[:, 1],
        station_boxes[:, 2],
        station_boxes[:, 3],
    )

    unique = {}
    for item in items:
        if not has_usable_polarizations(item):
            continue
        try:
            if not _item_near_any_sample_day(item, cluster_days):
                continue
            scene_geometry = _item_spatial_geometry(item)
            if scene_geometry is not None and not np.any(
                shapely.intersects(scene_geometry, station_geometries)
            ):
                continue
            unique[item.id] = item
        except (ValueError, TypeError, KeyError):
            continue
    return list(unique.values())


def _merged_sample_date_windows(cluster_stations, radius_days=MAX_NEAREST_DAYS):
    """Merge the union of [sample_day-radius, sample_day+radius] windows.

    The old strategy queried one continuous min_date..max_date interval for a
    cluster. If target dates were years apart, it downloaded metadata for all
    the dates in between. These merged windows include every date that can
    possibly contribute within the configured nearest-date tolerance, without
    querying those irrelevant gaps.
    """
    days = _cluster_sample_day_union(cluster_stations)
    if days is None or len(days) == 0:
        return []

    radius_days = int(radius_days)
    raw = sorted(
        (int(day) - radius_days, int(day) + radius_days)
        for day in np.asarray(days, dtype=np.int64)
    )
    merged = []
    for left, right in raw:
        if not merged or left > merged[-1][1] + 1:
            merged.append([left, right])
        else:
            merged[-1][1] = max(merged[-1][1], right)
    return [(int(left), int(right)) for left, right in merged]


def _query_cache_path(cluster_id, pair, start, end):
    """Stable cache key for one bounded STAC request."""
    start_day = pd.Timestamp(start).normalize().strftime("%Y%m%d")
    end_day = pd.Timestamp(end).normalize().strftime("%Y%m%d")
    pair_tag = "".join(str(x).upper() for x in pair)
    query_dir = STAC_CACHE_DIR / "query_pages"
    query_dir.mkdir(parents=True, exist_ok=True)
    return query_dir / (
        f"cluster_{int(cluster_id):05d}_{pair_tag}_{start_day}_{end_day}.json"
    )


def _cached_query_window_with_retry(bbox, start, end, cluster_id):
    """Resume a bounded window from cache; fall back to older paired-query caches."""
    all_path = _query_cache_path(cluster_id, ("ALL",), start, end)
    cached = load_stac_cache(all_path)
    if cached is not None:
        _emit_inventory_event(
            "query_cache_hit",
            cluster_id=int(cluster_id),
            start=str(pd.Timestamp(start).date()),
            end=str(pd.Timestamp(end).date()),
            items=len(cached),
            source="all-window cache",
        )
        return cached

    # Compatibility: if a previous run completed both polarization-specific
    # requests for exactly this window, combine them instead of downloading again.
    cached_vv = load_stac_cache(
        _query_cache_path(cluster_id, ("VV", "VH"), start, end)
    )
    cached_hh = load_stac_cache(
        _query_cache_path(cluster_id, ("HH", "HV"), start, end)
    )
    if cached_vv is not None and cached_hh is not None:
        by_id = {str(item.id): item for item in cached_vv}
        by_id.update({str(item.id): item for item in cached_hh})
        combined = list(by_id.values())
        save_stac_cache(all_path, combined)
        _emit_inventory_event(
            "query_cache_hit",
            cluster_id=int(cluster_id),
            start=str(pd.Timestamp(start).date()),
            end=str(pd.Timestamp(end).date()),
            items=len(combined),
            source="combined prior pair caches",
        )
        return combined

    items = _query_window_with_retry(bbox, start, end, cluster_id)
    save_stac_cache(all_path, items)
    return items


def search_cluster_full(cluster_row):
    """Search merged date windows; emit stages before doing potentially slow work."""
    cluster_id = int(cluster_row.cluster_id)
    _emit_inventory_event(
        "cluster_stage", cluster_id=cluster_id, stage="checking cluster cache"
    )
    try:
        new_cache = cluster_cache_path(cluster_id)
        cached = load_stac_cache(new_cache)
        if cached is not None:
            _emit_inventory_event(
                "cluster_stage", cluster_id=cluster_id,
                stage=f"cluster cache loaded ({len(cached):,} items)"
            )
            return cached

        _emit_inventory_event(
            "cluster_stage", cluster_id=cluster_id, stage="preparing cluster stations"
        )
        cluster_stations = stations_cluster[
            stations_cluster["cluster_id"] == cluster_id
        ].reset_index(drop=True)
        if cluster_stations.empty:
            save_stac_cache(new_cache, [])
            return []

        bbox = [
            float(cluster_row.bbox_min_lon),
            float(cluster_row.bbox_min_lat),
            float(cluster_row.bbox_max_lon),
            float(cluster_row.bbox_max_lat),
        ]
        _emit_inventory_event(
            "cluster_stage", cluster_id=cluster_id,
            stage="merging sample-date windows"
        )
        windows = _merged_sample_date_windows(cluster_stations)
        _emit_inventory_event(
            "cluster_stage", cluster_id=cluster_id,
            stage=f"date windows ready ({len(windows):,})"
        )
        if not windows:
            save_stac_cache(new_cache, [])
            return []

        def day_ts(day):
            return EPOCH + pd.Timedelta(days=int(day))

        found = {}
        for wi, (left_day, right_day) in enumerate(windows, start=1):
            _emit_inventory_event(
                "cluster_stage", cluster_id=cluster_id,
                stage=f"requesting date window {wi}/{len(windows)}"
            )
            # Inclusive target days are represented by an exclusive next-midnight end.
            window_items = _cached_query_window_with_retry(
                bbox,
                day_ts(left_day),
                day_ts(int(right_day) + 1),
                cluster_id,
            )
            for item in window_items:
                found[str(item.id)] = item

        _emit_inventory_event(
            "cluster_stage", cluster_id=cluster_id,
            stage=f"filtering {len(found):,} candidate scenes"
        )
        items = _filter_cluster_items(list(found.values()), cluster_row)
        items = sorted(items, key=lambda item: (item_day(item), str(item.id)))
        _emit_inventory_event(
            "cluster_stage", cluster_id=cluster_id,
            stage=f"writing cache ({len(items):,} relevant scenes)"
        )
        save_stac_cache(new_cache, items)
        return items
    finally:
        _emit_inventory_event("cluster_end", cluster_id=cluster_id)


def open_scene_inventory_db():
    conn = sqlite3.connect(
        SCENE_INDEX_DB_PATH,
        timeout=120,
    )
    conn.execute("PRAGMA journal_mode=WAL")
    conn.execute("PRAGMA synchronous=NORMAL")
    conn.execute("PRAGMA temp_store=MEMORY")
    conn.execute("PRAGMA busy_timeout=120000")
    conn.executescript(
        """
        CREATE TABLE IF NOT EXISTS scenes (
            scene_id TEXT PRIMARY KEY,
            day INTEGER NOT NULL,
            item_json TEXT NOT NULL
        );
        CREATE TABLE IF NOT EXISTS scene_clusters (
            scene_id TEXT NOT NULL,
            cluster_id INTEGER NOT NULL,
            PRIMARY KEY(scene_id, cluster_id)
        );
        CREATE TABLE IF NOT EXISTS cluster_searches (
            cluster_id INTEGER PRIMARY KEY,
            completed INTEGER NOT NULL,
            result_count INTEGER NOT NULL,
            completed_utc TEXT NOT NULL
        );
        CREATE INDEX IF NOT EXISTS idx_scenes_day ON scenes(day);
        CREATE INDEX IF NOT EXISTS idx_sc_cluster ON scene_clusters(cluster_id, scene_id);
        CREATE INDEX IF NOT EXISTS idx_sc_scene ON scene_clusters(scene_id, cluster_id);
        """
    )
    conn.commit()
    return conn


def _commit_cluster_items(conn, cluster_id, items):
    """Bulk-index a cluster and return the number of genuinely new scene rows."""
    if not items:
        with conn:
            conn.execute(
                """
                INSERT OR REPLACE INTO cluster_searches
                    (cluster_id, completed, result_count, completed_utc)
                VALUES (?, 1, 0, ?)
                """,
                (int(cluster_id), pd.Timestamp.now(tz="UTC").isoformat()),
            )
        return 0

    # Get existing IDs in safe-sized chunks. Avoid JSON serialization for the many
    # scene/cluster overlaps that have already been indexed by earlier clusters.
    unique_items = {str(item.id): item for item in items}
    ids = list(unique_items)
    existing_ids = set()
    for begin in range(0, len(ids), 800):
        batch = ids[begin:begin + 800]
        placeholders = ",".join(["?"] * len(batch))
        existing_ids.update(
            row[0]
            for row in conn.execute(
                f"SELECT scene_id FROM scenes WHERE scene_id IN ({placeholders})",
                batch,
            )
        )

    new_scene_rows = []
    for scene_id, item in unique_items.items():
        if scene_id in existing_ids:
            continue
        new_scene_rows.append((
            scene_id,
            item_day(item),
            json.dumps(item.to_dict(), separators=(",", ":")),
        ))

    association_rows = [
        (scene_id, int(cluster_id))
        for scene_id in unique_items
    ]

    with conn:
        if new_scene_rows:
            conn.executemany(
                "INSERT OR IGNORE INTO scenes(scene_id, day, item_json) VALUES (?, ?, ?)",
                new_scene_rows,
            )
        conn.executemany(
            "INSERT OR IGNORE INTO scene_clusters(scene_id, cluster_id) VALUES (?, ?)",
            association_rows,
        )
        conn.execute(
            """
            INSERT OR REPLACE INTO cluster_searches
                (cluster_id, completed, result_count, completed_utc)
            VALUES (?, 1, ?, ?)
            """,
            (
                int(cluster_id),
                len(items),
                pd.Timestamp.now(tz="UTC").isoformat(),
            ),
        )

    return len(new_scene_rows)


def build_global_scene_index(force=False):
    """Build the resumable date-window inventory with live request and scene heartbeats."""
    global _STAC_PROGRESS_EVENTS
    _STAC_PROGRESS_EVENTS = queue.Queue()

    cluster_rows = list(clusters.itertuples(index=False))
    all_ids = {int(row.cluster_id) for row in cluster_rows}
    conn = open_scene_inventory_db()

    if force:
        with conn:
            conn.execute("DELETE FROM scene_clusters")
            conn.execute("DELETE FROM scenes")
            conn.execute("DELETE FROM cluster_searches")

    completed = {
        int(row[0])
        for row in conn.execute(
            "SELECT cluster_id FROM cluster_searches WHERE completed = 1"
        )
    }
    pending = [row for row in cluster_rows if int(row.cluster_id) not in completed]
    existing_scene_count = int(
        conn.execute("SELECT COUNT(*) FROM scenes").fetchone()[0]
    )

    print("Clusters total:", f"{len(cluster_rows):,}")
    print("Clusters already complete:", f"{len(completed):,}")
    print("Clusters remaining:", f"{len(pending):,}")
    print("Unique scenes already indexed:", f"{existing_scene_count:,}")
    print("STAC search workers:", SEARCH_WORKERS)
    print("Scene workers:", SCENE_WORKERS)
    print("STAC request timeout:", STAC_REQUEST_TIMEOUT_SECONDS, "seconds")
    print("Window radius:", MAX_NEAREST_DAYS, "days around each sample date")

    if not pending:
        conn.close()
        _STAC_PROGRESS_EVENTS = None
        print("Inventory is already complete; no cluster searches to repeat.")
        return SCENE_INDEX_DB_PATH

    streaming_ready = STREAM_SCENES_DURING_INVENTORY and all(
        name in globals()
        for name in ("_scene_shard_path", "run_scene_record", "GLOBAL_STATION_TREE")
    )
    scene_executor = (
        ThreadPoolExecutor(max_workers=SCENE_WORKERS)
        if streaming_ready else None
    )
    max_scene_inflight = max(SCENE_WORKERS, min(SCENE_WORKERS * 2, 6))
    scene_futures = {}
    scene_started_at = {}
    scheduled_scene_ids = set()
    accounted_scene_ids = set()
    queued_scene_ids = set()
    scene_queue = deque()
    scene_failures = []
    scene_successes = 0
    running_scene_count = existing_scene_count

    # Store only IDs in the seed queue, not thousands of decoded item JSON objects.
    seed_scene_ids = deque(
        row[0] for row in conn.execute("SELECT scene_id FROM scenes ORDER BY rowid")
    ) if streaming_ready else deque()
    # Prevent scenes already in the initial backlog from also being queued again
    # when newly completed clusters report the same scene IDs.
    queued_scene_ids = set(seed_scene_ids)

    if streaming_ready:
        print(
            "Streaming extraction enabled: at most",
            max_scene_inflight,
            "scene jobs queued/in flight; one global spatial station index per scene.",
        )
    else:
        print("Scene extraction is decoupled from inventory; all missing shards are handled in stage 2.")

    active_queries = {}
    active_cluster_stages = {}
    inventory_started = time.monotonic()
    query_stats = {
        "started": 0,
        "finished": 0,
        "cache_hits": 0,
        "raw_items": 0,
        "failed": 0,
        "done": 0,
        "last_retry": "none",
        "last_event": time.monotonic(),
    }

    def drain_query_events(query_bar):
        while True:
            try:
                event = _STAC_PROGRESS_EVENTS.get_nowait()
            except queue.Empty:
                break

            kind = event.get("kind")
            query_stats["last_event"] = float(event.get("at", time.monotonic()))

            if kind == "cluster_stage":
                cid = int(event.get("cluster_id", -1))
                previous = active_cluster_stages.get(cid, {})
                active_cluster_stages[cid] = {
                    "stage": str(event.get("stage", "working")),
                    "started_at": previous.get(
                        "started_at", float(event.get("at", time.monotonic()))
                    ),
                }

            elif kind == "cluster_end":
                active_cluster_stages.pop(int(event.get("cluster_id", -1)), None)

            elif kind == "query_start":
                query_stats["started"] += 1
                active_queries[event["query_id"]] = {
                    **event,
                    "started_at": float(event.get("at", time.monotonic())),
                    "items_seen": 0,
                }
                query_bar.total = query_stats["started"] + query_stats["cache_hits"]
                query_bar.refresh()

            elif kind == "query_items":
                active = active_queries.get(event.get("query_id"))
                if active is not None:
                    active["items_seen"] = int(event.get("items_seen", 0))

            elif kind == "query_end":
                active_queries.pop(event.get("query_id"), None)
                query_stats["finished"] += 1
                query_stats["done"] += 1
                query_stats["raw_items"] += int(event.get("items", 0))
                if not event.get("success", False):
                    query_stats["failed"] += 1
                query_bar.update(1)

            elif kind == "query_cache_hit":
                query_stats["cache_hits"] += 1
                query_stats["done"] += 1
                query_stats["raw_items"] += int(event.get("items", 0))
                query_bar.total = query_stats["started"] + query_stats["cache_hits"]
                query_bar.update(1)

            elif kind == "query_retry":
                query_stats["last_retry"] = (
                    f"C{event.get('cluster_id')} "
                    f"attempt {event.get('attempt')}/{event.get('attempts')} "
                    f"in {event.get('delay')}s"
                )

    def update_live_bars(cluster_bar, query_bar, scene_bar, cluster_futures,
                         cluster_started_at, scene_started_at, scene_queue,
                         seed_scene_ids, running_scene_count, scene_successes):
        drain_query_events(query_bar)
        now = time.monotonic()

        active = sorted(
            active_queries.values(),
            key=lambda x: x.get("started_at", now),
        )
        if active:
            oldest_q = active[0]
            q_current = (
                f"C{oldest_q.get('cluster_id')} "
                f"{oldest_q.get('start')}..{oldest_q.get('end')}"
            )
            q_age = f"{int(now - oldest_q.get('started_at', now))}s"
            q_items = int(oldest_q.get("items_seen", 0))
        else:
            q_current, q_age, q_items = "no request active", "-", 0

        query_bar.set_postfix(
            active=len(active),
            current=q_current,
            age=q_age,
            items=q_items,
            raw=f"{query_stats['raw_items']:,}",
            cache=f"{query_stats['cache_hits']:,}",
            retries=query_stats["last_retry"],
            alive=f"{int(now - query_stats['last_event'])}s",
            refresh=True,
        )

        if cluster_futures:
            oldest_cluster_future = min(
                cluster_futures,
                key=lambda future: cluster_started_at.get(future, now),
            )
            oldest_row = cluster_futures[oldest_cluster_future]
            cluster_label = (
                f"C{int(oldest_row.cluster_id)} "
                f"{int(now - cluster_started_at.get(oldest_cluster_future, now))}s"
            )
        else:
            cluster_label = "none"

        stage_rows = sorted(
            active_cluster_stages.items(),
            key=lambda pair: pair[1].get("started_at", now),
        )
        stage_labels = [
            f"C{cid}:{entry.get('stage', 'working')}:{int(now - entry.get('started_at', now))}s"
            for cid, entry in stage_rows[:2]
        ]
        cluster_bar.set_postfix(
            checkpoints=f"{len(completed):,}/{len(all_ids):,}",
            active=len(cluster_futures),
            oldest=cluster_label,
            phase=" | ".join(stage_labels) or "no active stage",
            unique=f"{running_scene_count:,}",
            req=f"{query_stats['done']:,}/{query_stats['started'] + query_stats['cache_hits']:,}",
            refresh=True,
        )

        if scene_bar is not None:
            active_scene_rows = sorted(
                scene_futures.items(),
                key=lambda pair: scene_started_at.get(pair[0], now),
            )
            active_labels = []
            oldest_age = "-"
            for future, scene_id in active_scene_rows[:2]:
                age = int(now - scene_started_at.get(future, now))
                active_labels.append(f"{str(scene_id)[:16]}:{age}s")
            if active_scene_rows:
                oldest_age = f"{int(now - scene_started_at.get(active_scene_rows[0][0], now))}s"
            scene_bar.set_postfix(
                done=f"{scene_bar.n:,}/{int(scene_bar.total or 0):,}",
                discovered=f"{running_scene_count:,}",
                queued=len(scene_queue) + len(seed_scene_ids),
                active=len(scene_futures),
                current=", ".join(active_labels) or "none",
                oldest=oldest_age,
                elapsed=f"{int(now - inventory_started)}s",
                heartbeat="alive",
                refresh=True,
            )

    scene_bar = None
    cluster_bar = None
    query_bar = None
    pending_iter = iter(pending)
    cluster_futures = {}
    cluster_started_at = {}
    failures = []

    def submit_cluster(executor, row):
        future = executor.submit(search_cluster_full, row)
        cluster_futures[future] = row
        cluster_started_at[future] = time.monotonic()

    def account_scene(scene_id, bar):
        if scene_id in accounted_scene_ids:
            return
        accounted_scene_ids.add(scene_id)
        if bar is not None:
            bar.update(1)

    def pump_scene_jobs():
        if not streaming_ready:
            return
        checks_this_pass = 0
        max_checks_per_pass = 32
        while (
            len(scene_futures) < max_scene_inflight
            and checks_this_pass < max_checks_per_pass
        ):
            scene_id = None
            if scene_queue:
                scene_id = scene_queue.popleft()
                queued_scene_ids.discard(scene_id)
            elif seed_scene_ids:
                scene_id = seed_scene_ids.popleft()
                queued_scene_ids.discard(str(scene_id))
            else:
                break

            checks_this_pass += 1
            scene_id = str(scene_id)
            if not scene_id or scene_id in scheduled_scene_ids:
                continue

            if _scene_shard_path(scene_id).exists():
                scheduled_scene_ids.add(scene_id)
                account_scene(scene_id, scene_bar)
                continue

            row = conn.execute(
                "SELECT item_json FROM scenes WHERE scene_id = ?",
                (scene_id,),
            ).fetchone()
            if row is None:
                continue

            record = {
                "item": json.loads(row[0]),
                "cluster_ids": [],
                "global_scope": True,
            }
            future = scene_executor.submit(run_scene_record, record)
            scene_futures[future] = scene_id
            scene_started_at[future] = time.monotonic()
            scheduled_scene_ids.add(scene_id)

    def handle_scene_completions(done_futures):
        nonlocal scene_successes
        if not streaming_ready:
            return
        for future in list(done_futures):
            if future not in scene_futures:
                continue
            scene_id = scene_futures.pop(future)
            scene_started_at.pop(future, None)
            try:
                result = future.result()
                scene_successes += 1
                account_scene(scene_id, scene_bar)
                if scene_bar is not None:
                    scene_bar.set_postfix(
                        status=result.get("status", "ok"),
                        last=str(scene_id)[:18],
                        discovered=f"{running_scene_count:,}",
                        refresh=True,
                    )
            except KeyboardInterrupt:
                raise
            except Exception as exc:
                scene_failures.append((scene_id, f"{type(exc).__name__}: {exc}"))
                scheduled_scene_ids.discard(scene_id)
                tqdm.write(
                    f"Streaming scene {scene_id} failed (will retry in final extraction phase): "
                    f"{type(exc).__name__}: {exc}"
                )

    try:
        with ThreadPoolExecutor(max_workers=SEARCH_WORKERS) as search_executor:
            cluster_bar = tqdm(
                total=len(pending),
                desc="Inventory clusters checkpointed",
                unit="cluster",
                position=0,
                leave=True,
                dynamic_ncols=True,
                mininterval=0.5,
            )
            query_bar = tqdm(
                total=0,
                desc="STAC date-window requests",
                unit="req",
                position=1,
                leave=True,
                dynamic_ncols=True,
                mininterval=0.5,
            )
            if streaming_ready:
                scene_bar = tqdm(
                    total=existing_scene_count,
                    desc="Scene shards completed",
                    unit="scene",
                    position=2,
                    leave=True,
                    dynamic_ncols=True,
                    mininterval=0.5,
                )

            for _ in range(min(SEARCH_WORKERS, len(pending))):
                row = next(pending_iter, None)
                if row is not None:
                    submit_cluster(search_executor, row)

            while cluster_futures:
                update_live_bars(
                    cluster_bar, query_bar, scene_bar,
                    cluster_futures, cluster_started_at,
                    scene_started_at, scene_queue, seed_scene_ids,
                    running_scene_count, scene_successes,
                )
                watch = set(cluster_futures) | set(scene_futures)
                done_futures, _ = wait(
                    watch,
                    timeout=1.0,
                    return_when=FIRST_COMPLETED,
                )
                handle_scene_completions(done_futures)

                for future in list(done_futures):
                    if future not in cluster_futures:
                        continue
                    row = cluster_futures.pop(future)
                    cluster_started_at.pop(future, None)
                    cluster_id = int(row.cluster_id)
                    cluster_item_count = 0
                    try:
                        items = future.result()
                        cluster_item_count = len(items)
                        newly_indexed = _commit_cluster_items(conn, cluster_id, items)
                        running_scene_count += newly_indexed
                        completed.add(cluster_id)

                        if scene_bar is not None and newly_indexed:
                            scene_bar.total = int(scene_bar.total or 0) + newly_indexed
                            scene_bar.refresh()

                        if streaming_ready:
                            # Queue only scene IDs, not decoded STAC JSON. The JSON is
                            # fetched lazily only when a worker slot is ready.
                            for item in items:
                                scene_id = str(item.id)
                                if _scene_shard_path(scene_id).exists():
                                    scheduled_scene_ids.add(scene_id)
                                    account_scene(scene_id, scene_bar)
                                    continue
                                if (
                                    scene_id not in scheduled_scene_ids
                                    and scene_id not in queued_scene_ids
                                ):
                                    scene_queue.append(scene_id)
                                    queued_scene_ids.add(scene_id)
                    except KeyboardInterrupt:
                        raise
                    except Exception as exc:
                        failures.append(cluster_id)
                        tqdm.write(
                            f"Cluster {cluster_id} failed: {type(exc).__name__}: {exc}"
                        )

                    cluster_bar.update(1)
                    cluster_bar.set_postfix(
                        checkpointed=f"{len(completed):,}/{len(all_ids):,}",
                        unique_scenes=f"{running_scene_count:,}",
                        returned=cluster_item_count,
                        stream_ok=scene_successes,
                        refresh=True,
                    )

                    next_row = next(pending_iter, None)
                    if next_row is not None:
                        submit_cluster(search_executor, next_row)

                pump_scene_jobs()
                update_live_bars(
                    cluster_bar, query_bar, scene_bar,
                    cluster_futures, cluster_started_at,
                    scene_started_at, scene_queue, seed_scene_ids,
                    running_scene_count, scene_successes,
                )

            # Inventory searches are done. Drain all scene IDs discovered during
            # this run plus the database backlog from any interrupted earlier run.
            if streaming_ready:
                pump_scene_jobs()
                while scene_futures or scene_queue or seed_scene_ids:
                    if scene_futures:
                        done_futures, _ = wait(
                            list(scene_futures),
                            timeout=1.0,
                            return_when=FIRST_COMPLETED,
                        )
                        handle_scene_completions(done_futures)
                    pump_scene_jobs()
                    update_live_bars(
                        cluster_bar, query_bar, scene_bar,
                        cluster_futures, cluster_started_at,
                        scene_started_at, scene_queue, seed_scene_ids,
                        running_scene_count, scene_successes,
                    )

            # Drain final query status events before closing the progress bars.
            drain_query_events(query_bar)

    finally:
        for bar in (cluster_bar, query_bar, scene_bar):
            if bar is not None:
                bar.close()
        if scene_executor is not None:
            scene_executor.shutdown(wait=True)
        conn.close()
        _STAC_PROGRESS_EVENTS = None

    with sqlite3.connect(SCENE_INDEX_DB_PATH, timeout=120) as stats_conn:
        unique_count = int(
            stats_conn.execute("SELECT COUNT(*) FROM scenes").fetchone()[0]
        )
    with sqlite3.connect(SCENE_INDEX_DB_PATH) as stats_conn:
        association_count = int(
            stats_conn.execute("SELECT COUNT(*) FROM scene_clusters").fetchone()[0]
        )
        committed_count = int(
            stats_conn.execute(
                "SELECT COUNT(*) FROM cluster_searches WHERE completed=1"
            ).fetchone()[0]
        )

    print("Unique scenes:", f"{unique_count:,}")
    print("Scene-cluster associations:", f"{association_count:,}")
    print("Clusters committed:", f"{committed_count:,}/{len(all_ids):,}")
    print("Inventory database:", SCENE_INDEX_DB_PATH)
    if scene_failures:
        print(
            f"Streaming scene failures: {len(scene_failures)}. "
            "The final extraction stage will retry missing shards. "
            f"First failures: {scene_failures[:5]}"
        )
    if failures:
        raise RuntimeError(
            f"{len(failures)} cluster searches failed and remain resumable. "
            "Rerun the inventory cell after resolving the errors. "
            f"Failed cluster IDs (first 20): {[x for x in failures[:20]]}"
        )
    if completed != all_ids:
        raise RuntimeError(
            f"Inventory incomplete: {len(completed)} of {len(all_ids)} clusters committed."
        )
    return SCENE_INDEX_DB_PATH


def load_global_scene_index():
    """Compatibility alias; returns the SQLite index path, not a giant Python list."""
    return build_global_scene_index()


## Scene-centric production acquisition

The production unit is one unique Sentinel-1 RTC scene. Cluster queries are written incrementally to SQLite; repeated scenes are stored once and associated with every relevant cluster. The inventory uses merged intervals around actual sample dates, not one continuous min-to-max span, and caches each bounded interval atomically.

The inventory UI has three independent progress indicators: completed cluster checkpoints, individual STAC interval requests (including active request age, item counts, cache hits, and retries), and scene shards completed while inventory continues. A live scene heartbeat is shown while remote COG reads are in flight. A bounded queue keeps raster concurrency and memory use controlled. The STAC client uses a finite request timeout and retries.

For each sample, selection uses the closest valid acquisition within `MAX_NEAREST_DAYS` before or after the sample date; an equal-distance tie favors the earlier date. There is no temporal median.

In [11]:
# ============================================================
# SCENE SELECTION
# ============================================================

def item_day(item):
    dt = (
        item.datetime
        or item.properties.get("start_datetime")
        or item.properties.get("end_datetime")
    )
    if dt is None:
        raise ValueError(
            f"STAC item {item.id} has no acquisition datetime"
        )
    ts = pd.Timestamp(dt)
    if ts.tzinfo is not None:
        ts = ts.tz_convert(None)
    return int((ts.normalize() - EPOCH).days)


def available_polarizations(item):
    asset_names = {
        str(name).lower()
        for name in item.assets
    }
    return [pol for pol in REAL_BANDS if pol in asset_names]


def has_usable_polarizations(item):
    """Production extraction only uses VV/VH; HH/HV-only scenes are skipped."""
    return set(READ_POLARIZATIONS).issubset(set(available_polarizations(item)))


def has_required_polarizations(item):
    """VV/VH-only compatibility helper for diagnostics."""
    return {"vv", "vh"}.issubset(set(available_polarizations(item)))


def _item_spatial_geometry(item):
    """Return exact STAC geometry when present, otherwise its conservative bbox."""
    if item.geometry is not None:
        return shapely.geometry.shape(item.geometry)

    bbox = getattr(item, "bbox", None)
    if bbox is not None and len(bbox) >= 4:
        return shapely.box(
            float(bbox[0]), float(bbox[1]),
            float(bbox[2]), float(bbox[3]),
        )
    return None


def station_patch_intersects_item(item, station_df):
    item_geometry = _item_spatial_geometry(item)
    if item_geometry is None:
        # No usable footprint: retain the safe, conservative previous behavior.
        return np.ones(len(station_df), dtype=bool)

    boxes = station_patch_boxes_deg(station_df)
    geometries = shapely.box(
        boxes[:, 0],
        boxes[:, 1],
        boxes[:, 2],
        boxes[:, 3],
    )
    return np.asarray(
        shapely.intersects(item_geometry, geometries),
        dtype=bool,
    )


def _nearest_day_distance(days, target_day):
    """Minimum absolute day distance in a sorted int array."""
    if days is None or len(days) == 0:
        return np.inf
    k = int(np.searchsorted(days, target_day, side="left"))
    distance = np.inf
    if k < len(days):
        distance = min(distance, abs(int(days[k]) - int(target_day)))
    if k > 0:
        distance = min(distance, abs(int(days[k - 1]) - int(target_day)))
    return distance


def stations_needed_for_scene(item, station_df, sample_days):
    """
    Return stations whose spatial patch intersects the scene and which have
    at least one sample date within +/- MAX_NEAREST_DAYS of the acquisition.
    """
    if not has_usable_polarizations(item):
        return station_df.iloc[0:0].copy()

    spatial = station_patch_intersects_item(item, station_df)
    scene_day = item_day(item)
    station_ids = station_df["station_id"].astype(str).to_numpy()

    needed = np.zeros(len(station_df), dtype=bool)
    for j, station_id in enumerate(station_ids):
        if not spatial[j]:
            continue
        days = sample_days.get(station_id)
        if _nearest_day_distance(days, scene_day) <= MAX_NEAREST_DAYS:
            needed[j] = True

    return station_df.loc[needed].reset_index(drop=True)


## Search and scene selection

The full cluster query is filtered by the actual station-patch envelopes and sample dates. For production raster extraction, an RTC scene is retained only if at least one station can plausibly use it within the configured nearest-date tolerance and it provides the required VV/VH pair. HH/HV assets are not downloaded.

The final compiler checks actual raster validity and selects the closest usable scene. Exact temporal ties prefer the earlier acquisition.


In [12]:
# ============================================================
# NATIVE 10m WINDOWS
# ============================================================

_tls = threading.local()


def transformer_to(
    crs_string,
):
    cache = getattr(
        _tls,
        "transformers",
        None,
    )

    if cache is None:
        cache = {}
        _tls.transformers = cache

    if crs_string not in cache:

        cache[
            crs_string
        ] = Transformer.from_crs(
            "EPSG:4326",
            crs_string,
            always_xy=True,
        )

    return cache[
        crs_string
    ]


def station_patch_windows(
    transform,
    crs_string,
    width,
    height,
    station_df,
):
    """
    Exact 1300x1300 native-pixel windows
    centered on each station.
    """

    transformer = (
        transformer_to(
            crs_string
        )
    )

    x, y = transformer.transform(
        station_df[
            "longitude"
        ].to_numpy(float),

        station_df[
            "latitude"
        ].to_numpy(float),
    )

    inv = ~transform

    cols = (
        inv.a * x
        + inv.b * y
        + inv.c
    )

    rows = (
        inv.d * x
        + inv.e * y
        + inv.f
    )

    c0 = (
        np.floor(cols)
        .astype(np.int64)
        - HALF_PATCH_PX
    )

    r0 = (
        np.floor(rows)
        .astype(np.int64)
        - HALF_PATCH_PX
    )

    c1 = (
        c0
        + SOURCE_PATCH_PX
    )

    r1 = (
        r0
        + SOURCE_PATCH_PX
    )

    pixels = np.column_stack(
        [
            c0,
            r0,
            c1,
            r1,
        ]
    )

    valid = (
        (c1 > 0)
        &
        (r1 > 0)
        &
        (c0 < width)
        &
        (r0 < height)
    )

    return (
        pixels,
        valid,
    )

In [13]:
# ============================================================
# COG TILE PLANNER -- MEMORY-SAFE
# ============================================================

def scene_layout(src):
    block_h, block_w = src.block_shapes[0]
    if not src.profile.get("tiled", False):
        raise RuntimeError("Sentinel-1 RTC asset is unexpectedly not tiled.")
    return {
        "width": src.width,
        "height": src.height,
        "transform": src.transform,
        "crs": src.crs.to_string(),
        "block_w": int(block_w),
        "block_h": int(block_h),
        "dtype": src.dtypes[0],
        "nodata": src.nodata,
        "overviews": src.overviews(1),
        "key": (src.width, src.height, tuple(src.transform)[:6]),
    }


def merge_tile_boxes(tile_boxes, slack=MERGE_SLACK):
    """
    Greedy O(n log n) merge of tile-aligned boxes.

    The previous implementation built several n x n matrices. With many
    clusters contributing to one scene that can exhaust notebook RAM and
    kill the kernel. This version sorts boxes spatially and grows one local
    merge group at a time, so memory stays O(n).
    """
    boxes = np.asarray(tile_boxes, dtype=np.int64).reshape(-1, 4)
    if len(boxes) == 0:
        return boxes.copy(), []
    if len(boxes) == 1:
        return boxes.copy(), [[0]]

    order = np.lexsort((boxes[:, 1], boxes[:, 0], boxes[:, 3], boxes[:, 2]))
    sorted_boxes = boxes[order]

    merged = []
    groups = []
    cur = None
    cur_group = []
    cur_area_sum = 0

    for local_sorted_index, box in enumerate(sorted_boxes):
        x0, y0, x1, y1 = map(int, box)
        area = max(0, x1 - x0) * max(0, y1 - y0)

        if cur is None:
            cur = [x0, y0, x1, y1]
            cur_group = [int(order[local_sorted_index])]
            cur_area_sum = area
            continue

        ux0 = min(cur[0], x0)
        uy0 = min(cur[1], y0)
        ux1 = max(cur[2], x1)
        uy1 = max(cur[3], y1)
        union_area = max(0, ux1 - ux0) * max(0, uy1 - uy0)

        if union_area <= float(slack) * float(cur_area_sum + area):
            cur = [ux0, uy0, ux1, uy1]
            cur_group.append(int(order[local_sorted_index]))
            cur_area_sum += area
        else:
            merged.append(tuple(cur))
            groups.append(cur_group)
            cur = [x0, y0, x1, y1]
            cur_group = [int(order[local_sorted_index])]
            cur_area_sum = area

    merged.append(tuple(cur))
    groups.append(cur_group)

    return np.asarray(merged, dtype=np.int64), groups


def plan_scene_reads(layout, station_df):
    """Convert exact station patches to clipped, tile-aligned remote reads."""
    pixels, valid = station_patch_windows(
        layout["transform"],
        layout["crs"],
        layout["width"],
        layout["height"],
        station_df,
    )

    valid_indices = np.flatnonzero(valid)
    if not len(valid_indices):
        return {"layout": layout, "reads": []}

    p = pixels[valid_indices]
    bw = layout["block_w"]
    bh = layout["block_h"]

    tx0 = np.floor_divide(p[:, 0], bw)
    ty0 = np.floor_divide(p[:, 1], bh)
    tx1 = np.ceil(p[:, 2] / bw).astype(np.int64)
    ty1 = np.ceil(p[:, 3] / bh).astype(np.int64)

    max_tx = math.ceil(layout["width"] / bw)
    max_ty = math.ceil(layout["height"] / bh)

    tx0 = np.clip(tx0, 0, max_tx)
    ty0 = np.clip(ty0, 0, max_ty)
    tx1 = np.clip(tx1, 0, max_tx)
    ty1 = np.clip(ty1, 0, max_ty)

    tile_boxes = np.column_stack([tx0, ty0, tx1, ty1]).astype(np.int64)
    merged, groups = merge_tile_boxes(tile_boxes)

    station_ids = station_df["station_id"].to_numpy()
    reads = []

    for tile_box, group in zip(merged, groups):
        bx0, by0, bx1, by1 = map(int, tile_box)
        c0 = bx0 * bw
        r0 = by0 * bh
        c1 = min(bx1 * bw, layout["width"])
        r1 = min(by1 * bh, layout["height"])

        members = []
        for local_index in group:
            original_index = valid_indices[local_index]
            members.append(
                (
                    station_ids[original_index],
                    tuple(int(v) for v in pixels[original_index]),
                )
            )

        reads.append(
            {
                "window": Window(c0, r0, c1 - c0, r1 - r0),
                "members": members,
            }
        )

    return {"layout": layout, "reads": reads}


## Remote COG scene reading

Production defaults to direct 13 x 13 reads using COG overviews with average resampling. This is substantially faster in the representative benchmark, but it is approximate relative to averaging all native 10 m pixels. Change `PRODUCTION_READ_STRATEGY` in the configuration cell to `"exact"` to restore native 10 m reads and exact 100 x 100 pixel means.

The exact and overview strategies write to separate scene-shard and final-output folders. Do not manually copy shards across those folders.


In [14]:
# ============================================================
# REMOTE COG READ + 10m -> 1km AGGREGATION
# ============================================================


def _normalize_raster_values(data, nodata):
    """Convert nodata/non-finite values to NaN in a float32 array."""
    data = np.asarray(data, dtype=np.float32).copy()
    if nodata is not None:
        try:
            nodata_value = float(nodata)
            if not np.isnan(nodata_value):
                data[data == np.float32(nodata_value)] = np.nan
        except (TypeError, ValueError):
            pass
    data[~np.isfinite(data)] = np.nan
    return data


def read_block(src, window):
    """Read a native-resolution block for the exact aggregation strategy."""
    data = src.read(
        1,
        window=window,
        out_dtype="float32",
    )
    return _normalize_raster_values(data, src.nodata)


def extract_member_patch(block, read_window, requested_pixels):
    """Extract an exact native 10 m station patch from a merged read."""
    c0, r0, c1, r1 = map(int, requested_pixels)
    patch = np.full((SOURCE_PATCH_PX, SOURCE_PATCH_PX), np.nan, dtype=np.float32)

    read_c0 = max(c0, int(read_window.col_off))
    read_r0 = max(r0, int(read_window.row_off))
    read_c1 = min(c1, int(read_window.col_off + read_window.width))
    read_r1 = min(r1, int(read_window.row_off + read_window.height))
    if read_c1 <= read_c0 or read_r1 <= read_r0:
        return patch

    src_c0 = read_c0 - int(read_window.col_off)
    src_r0 = read_r0 - int(read_window.row_off)
    src_c1 = read_c1 - int(read_window.col_off)
    src_r1 = read_r1 - int(read_window.row_off)
    dst_c0 = read_c0 - c0
    dst_r0 = read_r0 - r0
    dst_c1 = dst_c0 + (read_c1 - read_c0)
    dst_r1 = dst_r0 + (read_r1 - read_r0)
    patch[dst_r0:dst_r1, dst_c0:dst_c1] = block[src_r0:src_r1, src_c0:src_c1]
    return patch


def pool_10m_to_1km(patch):
    """1300x1300 native pixels -> 13x13 means over 100x100 pixel cells."""
    x = patch.reshape(
        PATCH_SIZE,
        SOURCE_PIXELS_PER_MODEL_CELL,
        PATCH_SIZE,
        SOURCE_PIXELS_PER_MODEL_CELL,
    )
    valid = np.isfinite(x)
    count = valid.sum(axis=(1, 3))
    total = np.nansum(x, axis=(1, 3), dtype=np.float64)
    output = np.full((PATCH_SIZE, PATCH_SIZE), np.nan, dtype=np.float32)
    np.divide(total, count, out=output, where=(count > 0))
    return output


_DIRECT_STATE = {"enabled": None}   # None = not verified yet ("auto" keeps it OFF until verified)


def _direct_reads_enabled():
    if DIRECT_INSIDE_READS is True:
        return True
    if DIRECT_INSIDE_READS is False:
        return False
    return bool(_DIRECT_STATE["enabled"])


def _window_inside(c0, r0, c1, r1, width, height):
    return c1 > c0 and r1 > r0 and c0 >= 0 and r0 >= 0 and c1 <= width and r1 <= height


def read_overview_patch(src, requested_pixels, force_boundless=False, force_direct=False):
    """Read one station window directly to 13x13 using COG overviews.

    The 1300x1300 source window and 13x13 output preserve the station-grid
    alignment. Windows that overhang the raster are padded with NaN through a
    boundless read. This strategy is an approximation because the stored COG
    overviews were precomputed and need not equal a fresh native-pixel mean.

    Windows fully inside the raster are read directly: rasterio builds a VRT and
    re-opens the file by URL for every boundless read, which is expensive over
    HTTP. `force_boundless` / `force_direct` exist for verification only.
    """
    if not src.overviews(1):
        raise RuntimeError(
            "PRODUCTION_READ_STRATEGY='overview' but this COG asset has no "
            f"overviews: {src.name}"
        )

    c0, r0, c1, r1 = map(int, requested_pixels)
    window = Window(c0, r0, c1 - c0, r1 - r0)

    use_direct = (
        not force_boundless
        and (force_direct or _direct_reads_enabled())
        and _window_inside(c0, r0, c1, r1, src.width, src.height)
    )

    if use_direct:
        data = src.read(
            1,
            window=window,
            out_shape=(PATCH_SIZE, PATCH_SIZE),
            out_dtype="float32",
            resampling=Resampling.average,
        )
    else:
        data = src.read(
            1,
            window=window,
            out_shape=(PATCH_SIZE, PATCH_SIZE),
            out_dtype="float32",
            resampling=Resampling.average,
            boundless=True,
            fill_value=np.nan,
        )
    return _normalize_raster_values(data, src.nodata)


def read_remote_scene_band(item, asset_key, station_df, plan=None):
    """Read one RTC polarization using the configured exact/overview strategy."""
    for attempt in range(READ_RETRIES):
        try:
            # The token manager already refreshes proactively when expiry is near.
            # A generic Rasterio/network read failure is not evidence of an expired
            # SAS token; forcing a token API request on every retry can create a
            # token-request storm during transient blob-store trouble.
            href = rtc_token.sign(_clean_href(item.assets[asset_key].href))
            with rasterio.Env(**GDAL_ENV):
                with rasterio.open(href) as src:
                    layout = scene_layout(src)
                    if plan is None or plan["layout"]["key"] != layout["key"]:
                        plan = plan_scene_reads(layout, station_df)

                    station_patches = {}

                    if PRODUCTION_READ_STRATEGY == "overview":
                        # Only one 13x13 overview-backed result is requested per
                        # station. This avoids downloading the full native 1300x1300
                        # source patch and keeps the output grid explicitly aligned.
                        for read in plan["reads"]:
                            for station_id, requested_pixels in read["members"]:
                                station_patches[str(station_id)] = read_overview_patch(
                                    src,
                                    requested_pixels,
                                )
                    else:
                        # Exact path: preserve the existing merged tile reads and
                        # native 10 m -> 1 km aggregation.
                        for read in plan["reads"]:
                            block = read_block(src, read["window"])
                            for station_id, requested_pixels in read["members"]:
                                native_patch = extract_member_patch(
                                    block,
                                    read["window"],
                                    requested_pixels,
                                )
                                station_patches[str(station_id)] = pool_10m_to_1km(
                                    native_patch
                                )

                    return station_patches, plan

        except (
            requests.RequestException,
            rasterio.errors.RasterioIOError,
            OSError,
            TimeoutError,
            ConnectionError,
            RuntimeError,
        ) as exc:
            if attempt >= READ_RETRIES - 1:
                raise
            delay = min(15, 2 ** attempt)
            tqdm.write(
                f"Remote read failed for {item.id}/{asset_key} "
                f"(attempt {attempt + 1}/{READ_RETRIES}): "
                f"{type(exc).__name__}: {exc}. Retrying in {delay}s."
            )
            time.sleep(delay)

    raise RuntimeError(f"Remote read failed for {item.id}/{asset_key}.")


In [15]:
# ============================================================
# SCENE-CENTRIC REMOTE COG WORKER
# ============================================================

def _scene_station_subset(scene_cluster_ids):
    """Return unique station rows linked to this globally unique scene."""
    subset = stations_cluster[
        stations_cluster["cluster_id"].isin(scene_cluster_ids)
    ][
        ["station_id", "latitude", "longitude", "cluster_id"]
    ].copy()
    if subset.empty:
        return subset
    return subset.drop_duplicates("station_id").reset_index(drop=True)


def _global_scene_station_candidates(item):
    """Return only station-patch envelopes intersecting this scene's footprint."""
    geometry = _item_spatial_geometry(item)
    if geometry is None:
        # Conservative fallback only for malformed STAC items with no bbox.
        return GLOBAL_STATION_FRAME.copy()
    try:
        indices = np.asarray(
            GLOBAL_STATION_TREE.query(geometry, predicate="intersects")
        )
        if not np.issubdtype(indices.dtype, np.integer):
            # Compatibility fallback for older STRtree APIs.
            indices = np.flatnonzero(
                np.asarray(
                    shapely.intersects(geometry, GLOBAL_STATION_PATCH_GEOMETRIES),
                    dtype=bool,
                )
            )
    except (TypeError, AttributeError):
        indices = np.flatnonzero(
            np.asarray(
                shapely.intersects(geometry, GLOBAL_STATION_PATCH_GEOMETRIES),
                dtype=bool,
            )
        )
    if len(indices) == 0:
        return GLOBAL_STATION_FRAME.iloc[0:0].copy()
    return GLOBAL_STATION_FRAME.iloc[np.unique(indices)].reset_index(drop=True)


def process_scene_remote(item, scene_cluster_ids=None):
    """Read a scene once for every globally relevant station (or explicit clusters)."""
    if not has_usable_polarizations(item):
        return None

    if scene_cluster_ids is None:
        station_df = _global_scene_station_candidates(item)
    else:
        station_df = _scene_station_subset(scene_cluster_ids)
    if station_df.empty:
        return None

    needed_stations = stations_needed_for_scene(
        item,
        station_df,
        SAMPLE_DAYS,
    )
    if needed_stations.empty:
        return None

    station_ids = needed_stations["station_id"].astype(str).to_numpy()
    output = np.full(
        (
            len(station_ids),
            N_REAL_BANDS,
            PATCH_SIZE,
            PATCH_SIZE,
        ),
        np.nan,
        dtype=np.float32,
    )

    station_to_row = {
        station_id: i
        for i, station_id in enumerate(station_ids)
    }

    available = set(available_polarizations(item))
    bands_to_read = [
        (band_index, pol)
        for band_index, pol in enumerate(REAL_BANDS)
        if pol in READ_POLARIZATIONS and pol in available
    ]

    if PARALLEL_POLARIZATION_READS and len(bands_to_read) > 1:
        # VV and VH are the only remote assets read in this production mode.
        # Read them concurrently so a slow HTTP range request for one asset does not hold
        # up the independent asset read. Each worker opens its own Rasterio
        # dataset and uses the unchanged v9 window/resampling logic.
        # Persistent shared pool: the same threads (and their keep-alive HTTP
        # connections) are reused for every scene instead of new threads per scene.
        band_executor = _get_band_pool()
        future_to_band = {
            band_executor.submit(
                read_remote_scene_band,
                item,
                pol,
                needed_stations,
                None,
            ): (band_index, pol)
            for band_index, pol in bands_to_read
        }
        reference_layout_key = None
        for future in as_completed(future_to_band):
            band_index, pol = future_to_band[future]
            patches, band_plan = future.result()
            current_layout_key = band_plan["layout"]["key"]
            if reference_layout_key is None:
                reference_layout_key = current_layout_key
            elif current_layout_key != reference_layout_key:
                raise RuntimeError(
                    f"Polarization assets for scene {item.id} have mismatched grids; "
                    f"cannot combine {pol} with the other bands."
                )
            for station_id, patch in patches.items():
                output[station_to_row[str(station_id)], band_index] = patch
    else:
        # Sequential fallback reads only the configured VV/VH assets.
        plan = None
        for band_index, pol in bands_to_read:
            patches, plan = read_remote_scene_band(
                item,
                pol,
                needed_stations,
                plan=plan,
            )
            for station_id, patch in patches.items():
                output[station_to_row[str(station_id)], band_index] = patch

    return {
        "item_id": item.id,
        "day": item_day(item),
        "station_ids": station_ids,
        "data": output,
    }


def process_scene_with_retry(item, scene_cluster_ids, attempts=SCENE_RETRIES):
    """Retry transient remote raster failures; preserve completed scene shards."""
    for attempt in range(attempts):
        try:
            return process_scene_remote(
                item,
                scene_cluster_ids,
            )
        except KeyboardInterrupt:
            raise
        except (
            requests.RequestException,
            rasterio.errors.RasterioIOError,
            OSError,
            TimeoutError,
            ConnectionError,
        ) as exc:
            if attempt >= attempts - 1:
                raise
            delay = min(60, 2 ** (attempt + 1))
            tqdm.write(
                f"Scene {item.id} failed (attempt {attempt + 1}/{attempts}): "
                f"{type(exc).__name__}: {exc}. Retrying in {delay}s."
            )
            time.sleep(delay)
    return None


In [16]:
# ============================================================
# OPTIONAL REMOTE SMOKE TEST
# ============================================================

RUN_SMOKE_TEST = False

if RUN_SMOKE_TEST:
    test_cluster = clusters.sort_values(
        "n_stations",
        ascending=False,
    ).iloc[0]
    TEST_CLUSTER_ID = int(test_cluster["cluster_id"])
    TEST_STATIONS = stations_cluster[
        stations_cluster["cluster_id"] == TEST_CLUSTER_ID
    ].reset_index(drop=True)

    TEST_ITEMS = search_cluster_full(test_cluster)
    candidate_scenes = []
    for item in TEST_ITEMS:
        needed = stations_needed_for_scene(
            item,
            TEST_STATIONS,
            SAMPLE_DAYS,
        )
        if not needed.empty:
            candidate_scenes.append((len(needed), item, needed))

    if not candidate_scenes:
        raise RuntimeError(
            "No useful Sentinel-1 RTC scene in the test cluster can contribute "
            "to a station sample within the nearest-date tolerance."
        )

    _, TEST_ITEM, TEST_NEEDED_STATIONS = max(
        candidate_scenes,
        key=lambda x: x[0],
    )
    TEST_NEEDED_STATIONS = TEST_NEEDED_STATIONS.head(5).reset_index(drop=True)

    print("Test cluster:", TEST_CLUSTER_ID)
    print("Scene:", TEST_ITEM.id)
    print("Stations tested:", len(TEST_NEEDED_STATIONS))
    print("Available polarizations:", available_polarizations(TEST_ITEM))

    result = process_scene_remote(
        TEST_ITEM,
        [TEST_CLUSTER_ID],
    )
    if result is None:
        raise RuntimeError("Smoke test scene returned no station data.")

    print("Smoke-test output shape:", result["data"].shape)
    print("Expected final dimensions:", (len(result["station_ids"]), N_REAL_BANDS, PATCH_SIZE, PATCH_SIZE))
    print("Smoke test complete.")
else:
    print("Remote smoke test disabled; proceed to the global inventory cell when ready.")


Remote smoke test disabled; proceed to the global inventory cell when ready.


In [17]:
# ============================================================
# OPTIONAL REMOTE RANGE DIAGNOSTIC
# ============================================================

if "TEST_ITEM" not in globals():
    print("Skipping range diagnostic: TEST_ITEM is not defined. Enable the smoke test first if needed.")
else:
    available = [p for p in available_polarizations(TEST_ITEM) if p in READ_POLARIZATIONS]
    if not available:
        raise RuntimeError("TEST_ITEM has no required VV/VH polarization assets.")

    test_pol = available[0]
    test_href = rtc_token.sign(
        _clean_href(TEST_ITEM.assets[test_pol].href),
        force=True,
    )

    print("Testing a 1 KiB HTTP range from the remote COG URL...")
    response = requests.get(
        test_href,
        headers={"Range": "bytes=0-1023"},
        timeout=60,
    )

    print("Polarization:", test_pol)
    print("HTTP status:", response.status_code)
    print("Content-Type:", response.headers.get("Content-Type"))
    print("Content-Length:", response.headers.get("Content-Length"))
    print("Content-Range:", response.headers.get("Content-Range"))
    print("Bytes received:", len(response.content))


Skipping range diagnostic: TEST_ITEM is not defined. Enable the smoke test first if needed.


In [18]:
# ============================================================
# SCENE OBSERVATION CHECKPOINTS + NEAREST-VALID COMPILER
# ============================================================


def _scene_shard_path(scene_id):
    """Stable shard path; distributed workers write into disjoint subdirectories."""
    scene_id = str(scene_id)
    safe = re.sub(r"[^A-Za-z0-9_.-]+", "_", scene_id)
    suffix = hashlib.sha1(scene_id.encode("utf-8")).hexdigest()[:10]
    return _active_scene_shard_dir() / f"{safe[:140]}_{suffix}.npz"


def save_scene_observation(result):
    path = _scene_shard_path(result["item_id"])
    if path.exists():
        return path

    tmp_path = Path(str(path) + ".tmp")
    with tmp_path.open("wb") as f:
        np.savez_compressed(
            f,
            station_ids=np.asarray(result["station_ids"], dtype="U"),
            day=np.int64(result["day"]),
            data=result["data"].astype(np.float32, copy=False),
        )
        f.flush()
        os.fsync(f.fileno())
    os.replace(tmp_path, path)
    return path


def run_scene_record(record):
    """Idempotent unique-scene job, checkpointed as a compressed NPZ shard."""
    item = LightItem.from_dict(record["item"])
    shard = _scene_shard_path(item.id)

    if shard.exists():
        return {"scene_id": item.id, "status": "cached"}

    scene_cluster_ids = (
        None if record.get("global_scope", False)
        else record.get("cluster_ids", [])
    )
    result = process_scene_with_retry(
        item,
        scene_cluster_ids,
    )

    if result is None:
        result = {
            "item_id": item.id,
            "day": item_day(item),
            "station_ids": np.asarray([], dtype="U"),
            "data": np.empty(
                (0, N_REAL_BANDS, PATCH_SIZE, PATCH_SIZE),
                dtype=np.float32,
            ),
        }
        save_scene_observation(result)
        return {"scene_id": item.id, "status": "empty"}

    save_scene_observation(result)
    return {
        "scene_id": item.id,
        "status": "processed",
        "stations": len(result["station_ids"]),
    }


def _probe_planetary_computer():
    """Prime the shared SAS token before workers begin, failing fast if unavailable."""
    try:
        token = rtc_token.token(force=True)
        if not token:
            raise RuntimeError("Planetary Computer returned an empty SAS token.")
    except Exception as exc:
        raise RuntimeError(
            "Planetary Computer SAS-token preflight failed. No bulk scene reads were started. "
            "Check HTTPS connectivity, then rerun this stage; existing shards and the SQLite "
            f"inventory are preserved. Cause: {type(exc).__name__}: {exc}"
        ) from exc
    print("Planetary Computer SAS token acquired and cached before scene workers start.")
    return True

def _load_scene_observation(scene_id):
    path = _scene_shard_path(scene_id)
    if not path.exists():
        return None

    with np.load(path, allow_pickle=False) as z:
        return (
            z["station_ids"].astype(str),
            int(z["day"]),
            z["data"].astype(np.float32, copy=False),
        )


def compile_cluster_year_from_shards(cluster_id, year, samples_year):
    """
    Per sample, select the nearest scene with a valid complete pair:
      - VV/VH pair, and
      - HH/HV pair, independently.

    Nearest means absolute date difference, either before or after the label
    date, no farther than MAX_NEAREST_DAYS. Exact ties prefer earlier dates.
    """
    cluster_samples = samples_year[
        samples_year["cluster_id"] == cluster_id
    ].copy()

    if cluster_samples.empty:
        return None

    sample_days = cluster_samples["sample_date"].to_numpy(dtype="datetime64[D]").astype(np.int64)
    min_day = int(sample_days.min())
    max_day = int(sample_days.max())

    conn = sqlite3.connect(SCENE_INDEX_DB_PATH, timeout=120)
    scene_rows = conn.execute(
        """
        SELECT s.scene_id, s.day
        FROM scenes AS s
        JOIN scene_clusters AS sc ON sc.scene_id = s.scene_id
        WHERE sc.cluster_id = ?
          AND s.day >= ?
          AND s.day <= ?
        ORDER BY s.day, s.scene_id
        """,
        (
            int(cluster_id),
            min_day - MAX_NEAREST_DAYS,
            max_day + MAX_NEAREST_DAYS,
        ),
    ).fetchall()
    conn.close()

    # Build station -> sorted list of scene observations. Only the current
    # cluster's scene associations are loaded; the global inventory stays in SQLite.
    station_observations = {}
    for scene_id, expected_day in scene_rows:
        loaded = _load_scene_observation(scene_id)
        if loaded is None:
            raise FileNotFoundError(
                f"Missing scene shard {scene_id}; run scene extraction first."
            )

        station_ids, stored_day, data = loaded
        for i, station_id in enumerate(station_ids):
            station_observations.setdefault(station_id, []).append(
                (int(stored_day), str(scene_id), data[i])
            )

    for station_id in station_observations:
        station_observations[station_id].sort(
            key=lambda x: (x[0], x[1])
        )

    n = len(cluster_samples)
    output = np.full(
        (n, N_OUTPUT_BANDS, PATCH_SIZE, PATCH_SIZE),
        np.nan,
        dtype=np.float32,
    )
    observation_counts = np.zeros(
        (n, N_REAL_BANDS),
        dtype=np.uint8,
    )
    source_day_offset = np.full(
        (n, 2),
        SOURCE_OFFSET_MISSING,
        dtype=np.int16,
    )

    station_ids_for_samples = cluster_samples["station_id"].astype(str).to_numpy()
    pair_definitions = [
        (0, 1, 0),  # VV/VH, offset column 0
        (2, 3, 1),  # HH/HV, offset column 1
    ]

    for row, (station_id, sample_day) in enumerate(zip(station_ids_for_samples, sample_days)):
        entries = station_observations.get(station_id, [])
        if not entries:
            continue

        # Only candidates within the allowed nearest-date interval can be used.
        nearby = [
            (scene_day, scene_id, data)
            for scene_day, scene_id, data in entries
            if abs(int(scene_day) - int(sample_day)) <= MAX_NEAREST_DAYS
        ]
        if not nearby:
            continue

        for band_a, band_b, offset_index in pair_definitions:
            candidates = []

            for scene_day, scene_id, data in nearby:
                a = data[band_a]
                b = data[band_b]
                common_valid = np.isfinite(a) & np.isfinite(b)
                n_valid = int(common_valid.sum())

                if n_valid < MIN_VALID_PAIR_PIXELS:
                    continue

                delta = int(scene_day) - int(sample_day)
                # Rank by absolute distance, then prefer earlier, then more
                # jointly valid 1-km cells, then stable scene ID ordering.
                rank = (
                    abs(delta),
                    0 if delta <= 0 else 1,
                    -n_valid,
                    str(scene_id),
                )
                candidates.append((rank, scene_day, scene_id, data, n_valid))

            if not candidates:
                continue

            observation_counts[row, band_a] = min(len(candidates), 255)
            observation_counts[row, band_b] = min(len(candidates), 255)

            _, chosen_day, _, chosen_data, _ = min(
                candidates,
                key=lambda x: x[0],
            )

            output[row, band_a] = chosen_data[band_a]
            output[row, band_b] = chosen_data[band_b]
            source_day_offset[row, offset_index] = (
                int(chosen_day) - int(sample_day)
            )

    # Ratio is computed only after the aligned VV and VH observations have
    # been selected from the same scene/date.
    vv = output[:, 0]
    vh = output[:, 1]
    ratio = np.full_like(vv, np.nan)
    valid_ratio = np.isfinite(vv) & np.isfinite(vh) & (vh > 0)
    ratio[valid_ratio] = vv[valid_ratio] / vh[valid_ratio]
    output[:, OUTPUT_BANDS.index("vv_vh_ratio")] = ratio

    return output, observation_counts, source_day_offset


## Latency-optimised scene runtime

This cell keeps the same SQLite inventory and output shard paths, so completed v9 overview shards remain reusable. It overlaps VV/VH reads, keeps scene workers persistent, queues jobs with a bound, and writes shards on a separate pool.

**Evidence-based safety settings:** the saved direct-vs-boundless check failed on real data (23/23 patch values differed; 5 NaN masks differed), so direct reads are disabled. The saved run also started 48 scene workers (up to 96 band jobs) and then logged multiple range-read failures and SAS-token timeouts. The patched default starts at 18 scene workers (up to 36 band jobs), matching the previously tested throughput plateau.

The preflight now primes the shared SAS token once before bulk workers start, and a generic raster-read retry no longer forces a new token request. The token manager still refreshes as the token approaches expiry. The worker runner retains back-off and a lower-concurrency repass for failed scenes.


In [19]:
# ============================================================
# LATENCY-OPTIMISED SCENE RUNTIME (drop-in; same DB, same shards)
# ============================================================
#
# The job is bound by HTTP round trips, not bandwidth: ~204k scenes x 2
# polarizations, each only a handful of tiny 13x13 reads. This cell removes
# round trips and keeps more requests in flight:
#
#   1. Direct (non-boundless) reads were evaluated, but the saved real-data
#      comparison found value and NaN-mask mismatches. They remain disabled.
#   2. Persistent scene/band worker threads reduce Python scheduling churn.
#      Actual HTTP connection reuse depends on the installed GDAL/libcurl build.
#   3. A prefetching producer + bounded queue instead of a main loop that
#      submits one job per finished job (workers never wait for the scheduler).
#   4. Shard writing (compress + fsync) moved to a small writer pool, off the
#      network workers' critical path. Shard format/names are unchanged.
#   5. Adaptive concurrency: start high; if failures pile up (throttling or a
#      network blip) halve the worker limit, pause, then ramp back up. Scenes
#      that still fail get one automatic repass at lower concurrency.
#   6. One directory listing replaces 204k `Path.exists()` calls when resuming.

import logging
import statistics
import contextlib


# ------------------------------------------------------------
# Light scene item (no pystac.Item construction per scene)
# ------------------------------------------------------------

class _LightAsset:
    __slots__ = ("href",)

    def __init__(self, href):
        self.href = href


class LightItem:
    """The attributes the readers use: id, bbox, geometry, datetime, properties, assets."""

    __slots__ = ("id", "bbox", "geometry", "datetime", "properties", "assets")

    @classmethod
    def from_dict(cls, d):
        props = d.get("properties") or {}
        obj = cls()
        obj.id = d["id"]
        obj.bbox = d.get("bbox")
        obj.geometry = d.get("geometry")      # kept when stored, exactly as before
        obj.datetime = props.get("datetime")
        obj.properties = props
        obj.assets = {
            name: _LightAsset(asset["href"])
            for name, asset in (d.get("assets") or {}).items()
            if isinstance(asset, dict) and asset.get("href")
        }
        return obj


# ------------------------------------------------------------
# Persistent band-read pool (keeps one keep-alive connection per thread)
# ------------------------------------------------------------

_BAND_POOL = {"executor": None, "size": 0}
_BAND_POOL_LOCK = threading.Lock()


def configure_band_pool(scene_workers):
    """(Re)create the shared VV/VH reader pool sized for `scene_workers`."""
    size = max(2, int(scene_workers) * int(POLARIZATION_READ_WORKERS))
    with _BAND_POOL_LOCK:
        if _BAND_POOL["executor"] is None or _BAND_POOL["size"] != size:
            old = _BAND_POOL["executor"]
            _BAND_POOL["executor"] = ThreadPoolExecutor(
                max_workers=size, thread_name_prefix="band",
            )
            _BAND_POOL["size"] = size
            if old is not None:
                old.shutdown(wait=False)
    return _BAND_POOL["executor"]


def _get_band_pool():
    ex = _BAND_POOL["executor"]
    return ex if ex is not None else configure_band_pool(SCENE_WORKERS)


# ------------------------------------------------------------
# Scene job split: network work vs shard writing
# ------------------------------------------------------------

def _scene_job(record):
    """Read one scene. Returns (status, result) WITHOUT writing the shard."""
    item = LightItem.from_dict(record["item"])
    scene_cluster_ids = (
        None if record.get("global_scope", False)
        else record.get("cluster_ids", [])
    )
    result = process_scene_with_retry(item, scene_cluster_ids)

    if result is None:
        return "empty", {
            "item_id": item.id,
            "day": item_day(item),
            "station_ids": np.asarray([], dtype="U"),
            "data": np.empty(
                (0, N_REAL_BANDS, PATCH_SIZE, PATCH_SIZE), dtype=np.float32,
            ),
        }
    return "processed", result


# ------------------------------------------------------------
# Direct-read verification and latency diagnostics (real data; opt-in)
# ------------------------------------------------------------

def _probe_scenes_from_db(n=4):
    """A few inventoried scenes from across the timeline: [(LightItem, cluster_ids)]."""
    conn = sqlite3.connect(SCENE_INDEX_DB_PATH, timeout=120)
    try:
        total = conn.execute("SELECT COUNT(*) FROM scenes").fetchone()[0]
        picks = []
        for k in range(n):
            offset = int(total * (k + 1) / (n + 1))
            row = conn.execute(
                """
                SELECT s.scene_id, s.item_json,
                       (SELECT GROUP_CONCAT(cluster_id) FROM scene_clusters
                        WHERE scene_id = s.scene_id)
                FROM scenes AS s ORDER BY s.day, s.scene_id LIMIT 1 OFFSET ?
                """,
                (offset,),
            ).fetchone()
            if row:
                picks.append((
                    LightItem.from_dict(json.loads(row[1])),
                    [int(x) for x in str(row[2]).split(",") if x],
                ))
        return picks
    finally:
        conn.close()


def verify_direct_inside_reads(max_patches=24, min_patches=8, rtol=1e-4, atol=1e-6, verbose=True):
    """
    Compare direct reads against the original boundless (VRT) reads on real
    stations whose windows lie inside the raster. Both must agree on the NaN
    mask and on the values. Returns a dict; also sets the "auto" switch.
    """
    compared = 0
    max_abs = 0.0
    nan_mismatch = 0
    value_mismatch = 0
    scenes_used = 0
    reason = None

    try:
        for item, cluster_ids in _probe_scenes_from_db(n=5):
            station_df = stations_needed_for_scene(
                item, _scene_station_subset(cluster_ids), SAMPLE_DAYS,
            )
            if station_df.empty or "vv" not in item.assets:
                continue

            href = rtc_token.sign(_clean_href(item.assets["vv"].href))
            with rasterio.Env(**GDAL_ENV):
                with rasterio.open(href) as src:
                    layout = scene_layout(src)
                    plan = plan_scene_reads(layout, station_df)
                    scenes_used += 1
                    for read in plan["reads"]:
                        for _sid, req in read["members"]:
                            c0, r0, c1, r1 = map(int, req)
                            if not _window_inside(c0, r0, c1, r1, src.width, src.height):
                                continue
                            old = read_overview_patch(src, req, force_boundless=True)
                            new = read_overview_patch(src, req, force_direct=True)
                            compared += 1
                            same_nan = np.array_equal(np.isnan(old), np.isnan(new))
                            nan_mismatch += int(not same_nan)
                            both = np.isfinite(old) & np.isfinite(new)
                            if both.any():
                                max_abs = max(max_abs, float(np.abs(old[both] - new[both]).max()))
                            if not np.allclose(old, new, rtol=rtol, atol=atol, equal_nan=True):
                                value_mismatch += 1
                            if compared >= max_patches:
                                break
                        if compared >= max_patches:
                            break
            if compared >= max_patches:
                break
    except Exception as exc:                       # network/credentials/etc: stay safe
        reason = f"{type(exc).__name__}: {exc}"

    if reason is not None:
        passed = False
        verdict = f"inconclusive ({reason})"
    elif compared < min_patches:
        passed = False
        verdict = f"inconclusive (only {compared} comparable patches)"
    else:
        passed = (nan_mismatch == 0 and value_mismatch == 0)
        verdict = "PASS" if passed else "FAIL"

    _DIRECT_STATE["enabled"] = bool(passed)
    report = {
        "verdict": verdict, "passed": passed, "patches": compared, "scenes": scenes_used,
        "nan_mask_mismatches": nan_mismatch, "value_mismatches": value_mismatch,
        "max_abs_diff": max_abs,
    }
    if verbose:
        print(
            f"Direct-vs-boundless check: {verdict} | {compared} patches / {scenes_used} scenes | "
            f"NaN-mask mismatches {nan_mismatch} | value mismatches {value_mismatch} | "
            f"max |diff| {max_abs:.3g}"
        )
        if not passed:
            print("  -> direct reads stay DISABLED; the original boundless path is used.")
    return report


class _GetCounter(logging.Handler):
    """Counts ranged GETs that GDAL's /vsicurl logs at CPL_DEBUG level (URLs are never kept)."""

    def __init__(self):
        super().__init__(level=logging.DEBUG)
        self.requests = 0
        self.bytes = 0

    def emit(self, record):
        try:
            msg = record.getMessage()
        except Exception:
            return
        if "VSICURL: Downloading" in msg:
            self.requests += 1
            m = re.search(r"Downloading (\d+)-(\d+)", msg)
            if m:
                self.bytes += int(m.group(2)) - int(m.group(1)) + 1


def measure_http_latency(href=None, repeats=6):
    """Cold (new connection) vs warm (keep-alive) latency of a 1-byte ranged GET."""
    if href is None:
        item, _ = _probe_scenes_from_db(n=1)[0]
        href = item.assets["vv"].href
    url = rtc_token.sign(_clean_href(href))

    cold = []
    for _ in range(3):
        with requests.Session() as s:
            t = time.perf_counter()
            r = s.get(url, headers={"Range": "bytes=0-0"}, timeout=30)
            r.close()
            cold.append(time.perf_counter() - t)
    warm = []
    with requests.Session() as s:
        s.get(url, headers={"Range": "bytes=0-0"}, timeout=30).close()
        for _ in range(repeats):
            t = time.perf_counter()
            s.get(url, headers={"Range": "bytes=0-0"}, timeout=30).close()
            warm.append(time.perf_counter() - t)

    c, w = statistics.median(cold), statistics.median(warm)
    print(f"1-byte ranged GET  cold (new connection): {c * 1000:6.0f} ms | warm (keep-alive): {w * 1000:6.0f} ms")
    print(f"  => a connection that is not reused costs ~{(c - w) * 1000:.0f} ms extra per read")
    return {"cold_s": c, "warm_s": w}


def profile_scene_latency(n_scenes=2):
    """
    Per scene/band: seconds, ranged GETs and MB for the ORIGINAL boundless
    path versus the DIRECT path, on real data. Nothing is written to disk.
    """
    rows = []
    logger = logging.getLogger("rasterio._env")
    old_level = logger.level
    counter = _GetCounter()
    logger.addHandler(counter)
    logger.setLevel(logging.DEBUG)
    try:
        for item, cluster_ids in _probe_scenes_from_db(n=n_scenes + 1)[:n_scenes]:
            station_df = stations_needed_for_scene(
                item, _scene_station_subset(cluster_ids), SAMPLE_DAYS,
            )
            if station_df.empty:
                continue
            for mode in ("boundless (original)", "direct"):
                for _warm in (False, True):
                    counter.requests = counter.bytes = 0
                    href = rtc_token.sign(_clean_href(item.assets["vv"].href))
                    t = time.perf_counter()
                    with rasterio.Env(CPL_DEBUG="ON", **GDAL_ENV):
                        with rasterio.open(href) as src:
                            layout = scene_layout(src)
                            plan = plan_scene_reads(layout, station_df)
                            n = 0
                            for read in plan["reads"]:
                                for _sid, req in read["members"]:
                                    read_overview_patch(
                                        src, req,
                                        force_boundless=(mode != "direct"),
                                        force_direct=(mode == "direct"),
                                    )
                                    n += 1
                    rows.append({
                        "scene": item.id[:24], "mode": mode,
                        "run": "2nd (warm)" if _warm else "1st", "stations": n,
                        "seconds": round(time.perf_counter() - t, 2),
                        "GETs": counter.requests, "MB": round(counter.bytes / 1e6, 2),
                    })
    finally:
        logger.removeHandler(counter)
        logger.setLevel(old_level)
    df = pd.DataFrame(rows)
    print(df.to_string(index=False))
    return df


# ------------------------------------------------------------
# Adaptive concurrency gate
# ------------------------------------------------------------

class AdaptiveGate:
    """A resizable concurrency limit with a global pause (for back-off)."""

    def __init__(self, limit, minimum):
        self._cv = threading.Condition()
        self.minimum = max(1, int(minimum))
        self.limit = max(self.minimum, int(limit))
        self.active = 0
        self._pause_until = 0.0

    def acquire(self, stop):
        with self._cv:
            while True:
                if stop.is_set():
                    return False
                now = time.monotonic()
                if now >= self._pause_until and self.active < self.limit:
                    self.active += 1
                    return True
                self._cv.wait(timeout=0.25)

    def release(self):
        with self._cv:
            self.active -= 1
            self._cv.notify_all()

    def set_limit(self, n):
        with self._cv:
            self.limit = max(self.minimum, int(n))
            self._cv.notify_all()

    def pause(self, seconds):
        with self._cv:
            self._pause_until = max(self._pause_until, time.monotonic() + float(seconds))


# ------------------------------------------------------------
# Deterministic distributed ownership: each scene belongs to one worker.
# This is workload partitioning, not a mechanism to bypass rate limits.
# ------------------------------------------------------------

def _distributed_scene_owner(scene_id):
    digest = hashlib.sha256(str(scene_id).encode("utf-8")).digest()
    return int.from_bytes(digest[:8], "big") % int(DISTRIBUTED_WORKER_COUNT)


def _load_distributed_completed_ids():
    """Scene IDs already extracted on the coordinator; their shards stay there."""
    if not DISTRIBUTED_SCENE_WORKER_MODE:
        return set()
    manifest = Path(DISTRIBUTED_COMPLETED_SCENES_FILE)
    if not manifest.exists():
        raise FileNotFoundError(
            f"Missing {manifest}. On the coordinator, finish the inventory and run "
            "export_distributed_worker_package() first, then copy/unzip the package on each worker."
        )
    return {line.strip() for line in manifest.read_text(encoding="utf-8").splitlines() if line.strip()}


# ------------------------------------------------------------
# One pass over a stream of scene jobs
# ------------------------------------------------------------

_END = object()


def _iter_db_jobs(db_path, existing_names, counters, stop):
    """Yield assigned scenes only; safely skip coordinator and worker checkpoints."""
    conn = sqlite3.connect(db_path, timeout=120)
    completed_ids = DISTRIBUTED_COMPLETED_SCENE_IDS if DISTRIBUTED_SCENE_WORKER_MODE else set()
    try:
        cursor = conn.execute(
            """
            SELECT s.scene_id, GROUP_CONCAT(sc.cluster_id)
            FROM scenes AS s
            JOIN scene_clusters AS sc ON sc.scene_id = s.scene_id
            GROUP BY s.scene_id
            ORDER BY s.day, s.scene_id
            """
        )
        for scene_id, cluster_ids_raw in cursor:
            if stop.is_set():
                return
            scene_id = str(scene_id)
            if DISTRIBUTED_SCENE_WORKER_MODE:
                if _distributed_scene_owner(scene_id) != DISTRIBUTED_WORKER_ID:
                    continue
                # These shards already exist on the coordinator. Don't re-read them;
                # the master keeps the actual files and only the IDs are distributed.
                if scene_id in completed_ids:
                    counters["cached"] += 1
                    continue
            if _scene_shard_path(scene_id).name in existing_names:
                counters["cached"] += 1
                continue
            row = conn.execute(
                "SELECT item_json FROM scenes WHERE scene_id = ?", (scene_id,),
            ).fetchone()
            if row is None:
                continue
            yield scene_id, {
                "item": json.loads(row[0]),
                "cluster_ids": [int(x) for x in str(cluster_ids_raw).split(",") if x],
            }
    finally:
        conn.close()


def _run_scene_pass(jobs, workers, streak_limit, progress, counters, stop, label="pass"):
    """
    Process a stream of (scene_id, record) with persistent workers.
    Returns (failed {scene_id: record}, abort_reason | None, completed_count).
    """
    workers = max(1, int(workers))
    min_workers = min(workers, max(1, int(MIN_SCENE_WORKERS)))
    gate = AdaptiveGate(workers, min_workers)
    work_q = queue.Queue(maxsize=max(workers * int(SCENE_QUEUE_FACTOR), workers + 1))
    results_q = queue.Queue()
    writer = ThreadPoolExecutor(
        max_workers=max(1, int(SHARD_WRITER_WORKERS)), thread_name_prefix="shard",
    )
    write_slots = threading.BoundedSemaphore(workers * 2 + int(SHARD_WRITER_WORKERS))

    enq = {"n": 0}
    producer_done = threading.Event()
    producer_error = []

    def producer():
        try:
            for job in jobs:
                while not stop.is_set():
                    try:
                        work_q.put(job, timeout=0.5)
                        enq["n"] += 1
                        break
                    except queue.Full:
                        continue
                if stop.is_set():
                    break
        except Exception as exc:
            producer_error.append(exc)
        finally:
            producer_done.set()
            for _ in range(workers):
                while True:
                    try:
                        work_q.put(_END, timeout=0.5)
                        break
                    except queue.Full:
                        if stop.is_set():
                            break

    def write_done(scene_id, status, n_stations, record, fut):
        write_slots.release()
        exc = fut.exception()
        if exc is not None:
            results_q.put(("err", scene_id, (exc, record)))
        else:
            results_q.put(("ok", scene_id, (status, n_stations)))

    def worker():
        while True:
            try:
                job = work_q.get(timeout=0.5)
            except queue.Empty:
                if stop.is_set():
                    return
                continue
            if job is _END:
                return
            if not gate.acquire(stop):
                return
            scene_id, record = job
            try:
                status, result = _scene_job(record)
            except Exception as exc:
                results_q.put(("err", scene_id, (exc, record)))
                continue
            finally:
                gate.release()

            write_slots.acquire()
            try:
                fut = writer.submit(save_scene_observation, result)
            except Exception as exc:
                write_slots.release()
                results_q.put(("err", scene_id, (exc, record)))
                continue
            n_stations = int(len(result["station_ids"]))
            fut.add_done_callback(
                lambda f, sid=scene_id, st=status, ns=n_stations, rec=record:
                    write_done(sid, st, ns, rec, f)
            )

    threads = [threading.Thread(target=producer, name="scene-producer", daemon=True)]
    threads += [
        threading.Thread(target=worker, name=f"scene-{i}", daemon=True)
        for i in range(workers)
    ]
    for t in threads:
        t.start()

    failed = {}
    abort_reason = None
    finished = 0
    successes = 0
    streak = 0
    last_cached = 0
    t0 = time.monotonic()
    ramp_every = max(20, workers)

    try:
        while True:
            if progress is not None and counters["cached"] > last_cached:
                progress.update(counters["cached"] - last_cached)
                last_cached = counters["cached"]

            if producer_done.is_set() and finished >= enq["n"]:
                break
            if stop.is_set() and finished >= enq["n"]:
                break

            try:
                kind, scene_id, payload = results_q.get(timeout=2.0)
            except queue.Empty:
                if progress is not None:
                    progress.set_postfix(
                        active=gate.active, limit=gate.limit, queued=work_q.qsize(),
                        failed=len(failed), elapsed=f"{(time.monotonic() - t0) / 60:.1f}m",
                        refresh=True,
                    )
                continue

            finished += 1
            if progress is not None:
                progress.update(1)

            if kind == "ok":
                status, n_stations = payload
                successes += 1
                streak = 0
                counters["new"] += 1
                if (
                    gate.limit < workers
                    and successes % ramp_every == 0
                    and abort_reason is None
                ):
                    gate.set_limit(min(workers, gate.limit + max(1, workers // 12)))
                if progress is not None:
                    progress.set_postfix(
                        status=status, last=str(scene_id)[:18], active=gate.active,
                        limit=gate.limit, queued=work_q.qsize(), failed=len(failed),
                        refresh=False,
                    )
            else:
                exc, record = payload
                failed[str(scene_id)] = record
                streak += 1
                tqdm.write(
                    f"Scene {scene_id} failed ({streak}/{streak_limit} consecutive): "
                    f"{type(exc).__name__}: {exc}"
                )
                if streak >= streak_limit and abort_reason is None:
                    if gate.limit > gate.minimum:
                        new_limit = max(gate.minimum, gate.limit // 2)
                        gate.set_limit(new_limit)
                        gate.pause(PC_TOKEN_COOLDOWN_SECONDS)
                        streak = 0
                        tqdm.write(
                            f"[{label}] {streak_limit} consecutive failures: backing off to "
                            f"{new_limit} concurrent scenes and pausing "
                            f"{PC_TOKEN_COOLDOWN_SECONDS}s (throttling or a network blip)."
                        )
                    else:
                        abort_reason = (
                            f"Stopped after {streak} consecutive scene failures even at the "
                            f"minimum concurrency ({gate.minimum}). This indicates a remote "
                            "service/network problem or a repeated asset-read error."
                        )
                        tqdm.write(abort_reason)
                        stop.set()
                        break
    except KeyboardInterrupt:
        stop.set()
        tqdm.write("Interrupted: finishing in-flight scenes and flushing shard writes...")
        raise
    finally:
        if abort_reason is not None:
            stop.set()
        for t in threads:
            t.join(timeout=120)
        writer.shutdown(wait=True)       # every completed scene reaches disk
        if progress is not None and counters["cached"] > last_cached:
            progress.update(counters["cached"] - last_cached)

    # results that landed while we were shutting down
    while True:
        try:
            kind, scene_id, payload = results_q.get_nowait()
        except queue.Empty:
            break
        finished += 1
        if kind == "ok":
            counters["new"] += 1
        else:
            failed[str(scene_id)] = payload[1]

    if producer_error:
        raise RuntimeError(f"Scene job producer failed: {producer_error[0]!r}")

    return failed, abort_reason, finished


def run_scene_extraction_from_db(db_path=None, workers=None, failure_streak_limit=None):
    """Resume missing shards with persistent workers, adaptive concurrency and a repass."""
    db_path = SCENE_INDEX_DB_PATH if db_path is None else db_path
    workers = max(1, int(SCENE_WORKERS if workers is None else workers))
    streak_limit = max(
        1, int(SCENE_FAILURE_STREAK_LIMIT if failure_streak_limit is None else failure_streak_limit)
    )
    _probe_planetary_computer()
    configure_band_pool(workers)

    if DIRECT_INSIDE_READS == "auto" and _DIRECT_STATE["enabled"] is None:
        print("Verifying direct (non-boundless) reads against the original path on real data...")
        verify_direct_inside_reads()

    global DISTRIBUTED_COMPLETED_SCENE_IDS
    if DISTRIBUTED_SCENE_WORKER_MODE:
        DISTRIBUTED_COMPLETED_SCENE_IDS = _load_distributed_completed_ids()

    conn = sqlite3.connect(db_path, timeout=120)
    if DISTRIBUTED_SCENE_WORKER_MODE:
        owner_cursor = conn.execute(
            """SELECT s.scene_id FROM scenes AS s
               JOIN scene_clusters AS sc ON sc.scene_id = s.scene_id
               GROUP BY s.scene_id"""
        )
        total = sum(
            1 for (sid,) in owner_cursor
            if _distributed_scene_owner(sid) == DISTRIBUTED_WORKER_ID
        )
    else:
        total = int(conn.execute("SELECT COUNT(*) FROM scenes").fetchone()[0])
    conn.close()

    active_dir = _active_scene_shard_dir()
    active_dir.mkdir(parents=True, exist_ok=True)
    existing_names = set(os.listdir(active_dir))
    counters = {"cached": 0, "new": 0}
    stop = threading.Event()
    run_started = time.monotonic()

    if DISTRIBUTED_SCENE_WORKER_MODE:
        print(
            f"Distributed worker {DISTRIBUTED_WORKER_ID + 1}/{DISTRIBUTED_WORKER_COUNT}; "
            f"assigned scenes (including coordinator-completed IDs): {total:,}"
        )
        print("Worker shard directory:", active_dir)
        print("Coordinator-completed scenes skipped:", f"{len(DISTRIBUTED_COMPLETED_SCENE_IDS):,} total in manifest")
    else:
        print("Unique scenes to check:", f"{total:,}")
    print("Scene workers (start):", workers, "| band-read threads:", _BAND_POOL["size"])
    print("Direct inside-raster reads:", "ON" if _direct_reads_enabled() else "OFF")
    print("Shard writer threads:", SHARD_WRITER_WORKERS)
    print("GDAL HTTP timeout:", GDAL_ENV["GDAL_HTTP_TIMEOUT"], "seconds per HTTP operation")
    print("Back-off: halve concurrency after", streak_limit, "consecutive failures (min",
          min(workers, MIN_SCENE_WORKERS), ")")

    failed, abort_reason = {}, None
    with tqdm(
        total=total, desc="Scene shards completed", unit="scene",
        dynamic_ncols=True, mininterval=0.5,
    ) as progress:
        jobs = _iter_db_jobs(db_path, existing_names, counters, stop)
        failed, abort_reason, _ = _run_scene_pass(
            jobs, workers, streak_limit, progress, counters, stop, label="main",
        )

    if failed and abort_reason is None and SCENE_FAILED_REPASS:
        repass_workers = max(1, min(workers, max(MIN_SCENE_WORKERS, workers // 2)))
        print(f"\nRepass: retrying {len(failed):,} failed scenes at {repass_workers} workers...")
        before = len(failed)
        stop2 = threading.Event()
        failed2, abort_reason, _ = _run_scene_pass(
            iter(list(failed.items())), repass_workers, streak_limit, None,
            counters, stop2, label="repass",
        )
        print(f"Repass recovered {before - len(failed2):,} of {before:,} scenes.")
        failed = failed2

    present = counters["cached"] + counters["new"]
    elapsed_min = (time.monotonic() - run_started) / 60

    if abort_reason is not None:
        raise RuntimeError(
            f"{abort_reason} Completed/cached shards: {present:,}/{total:,}. "
            "The database and every completed shard are preserved. Fix connectivity or "
            "inspect the first failure, then rerun this stage to resume."
        )
    if failed:
        raise RuntimeError(
            f"{len(failed)} scene jobs still failed after the repass. Successful shards are "
            f"preserved; rerun this stage to retry. First failed IDs: {list(failed)[:20]}"
        )

    print("Scene shards present:", f"{present:,}/{total:,}")
    print(f"Cached shards reused: {counters['cached']:,}")
    print(f"New scene jobs completed: {counters['new']:,}")
    print(f"Scene extraction elapsed: {elapsed_min:.1f} minutes")
    if counters["new"]:
        print(f"Throughput: {counters['new'] / max(elapsed_min * 60, 1e-9):.2f} scenes/s")


In [20]:
# ============================================================
# YEARLY ZARR
# ============================================================

def create_year_zarr(path, n_samples):
    compressor = BloscCodec(
        cname="zstd",
        clevel=3,
        shuffle=BloscShuffle.bitshuffle,
    )

    root = zarr.open_group(path, mode="a")

    if "sample_done" not in root:
        root.attrs.update(
            {
                "schema_version": SCHEMA_VERSION,
                "production_read_strategy": PRODUCTION_READ_STRATEGY,
                "aggregation_method": (
                    "COG overview average resampling to 13x13; approximate"
                    if PRODUCTION_READ_STRATEGY == "overview"
                    else "native 10 m reads and exact 100x100 pixel means"
                ),
                "collection": S1_COLLECTION,
                "patch_size": PATCH_SIZE,
                "model_resolution_m": MODEL_RESOLUTION_M,
                "source_resolution_m": S1_RESOLUTION_M,
                "max_nearest_days": MAX_NEAREST_DAYS,
                "real_bands": REAL_BANDS,
                "output_bands": OUTPUT_BANDS,
                "temporal_selection": (
                    "nearest valid acquisition by absolute date difference; "
                    "earlier date wins exact ties; VV/VH and HH/HV are selected independently"
                ),
                "source_day_offset_columns": [
                    "vv_vh_offset_days",
                    "hh_hv_offset_days",
                ],
                "source_day_offset_missing_value": int(SOURCE_OFFSET_MISSING),
                "vv_vh_ratio": "selected_scene_VV / selected_scene_VH",
                "storage": (
                    "native RTC COG range reads + local 10m-to-1km aggregation "
                    "+ nearest-valid scene selection"
                ),
            }
        )

        root.create_array(
            "sample_done",
            shape=(n_samples,),
            chunks=(min(4096, max(1, n_samples)),),
            dtype="bool",
            fill_value=False,
        )

        for band in OUTPUT_BANDS:
            root.create_array(
                band,
                shape=(n_samples, PATCH_SIZE, PATCH_SIZE),
                chunks=(256, PATCH_SIZE, PATCH_SIZE),
                dtype="float32",
                fill_value=np.nan,
                compressors=compressor,
            )

        root.create_array(
            "obs_count",
            shape=(n_samples, N_REAL_BANDS),
            chunks=(1024, N_REAL_BANDS),
            dtype="uint8",
            fill_value=0,
            compressors=None,
        )

        root.create_array(
            "source_day_offset",
            shape=(n_samples, 2),
            chunks=(1024, 2),
            dtype="int16",
            fill_value=SOURCE_OFFSET_MISSING,
            compressors=None,
        )

    else:
        existing_schema = root.attrs.get("schema_version")
        if existing_schema != SCHEMA_VERSION:
            raise RuntimeError(
                f"Existing Zarr at {path} has schema_version={existing_schema}; "
                f"expected {SCHEMA_VERSION}. Use the versioned nearest_valid_v5 directory."
            )

        if root["sample_done"].shape[0] != n_samples:
            raise RuntimeError("Existing Zarr has a different number of samples.")

        expected_patch_shape = (n_samples, PATCH_SIZE, PATCH_SIZE)
        for band in OUTPUT_BANDS:
            if band not in root or tuple(root[band].shape) != expected_patch_shape:
                raise RuntimeError(
                    f"Existing Zarr band {band!r} is missing or has an incompatible shape."
                )

        if "obs_count" not in root or tuple(root["obs_count"].shape) != (n_samples, N_REAL_BANDS):
            raise RuntimeError("Existing Zarr obs_count has an incompatible shape.")

        if "source_day_offset" not in root or tuple(root["source_day_offset"].shape) != (n_samples, 2):
            raise RuntimeError("Existing Zarr source_day_offset has an incompatible shape.")

    # This attribute is updated on both new and resumed stores to make the current
    # extraction scope explicit; it does not invalidate existing completed samples.
    root.attrs["remote_read_polarizations"] = list(READ_POLARIZATIONS)
    root.attrs["hh_hv_remote_reads_disabled"] = True
    return root


In [21]:
# ============================================================
# YEARLY FINALIZATION
# ============================================================

def prepare_year_metadata(year):
    result = (
        samples_c[samples_c["year"] == year]
        .copy()
        .sort_values(
            ["cluster_id", "station_id", "sample_date"],
            kind="stable",
        )
        .reset_index(drop=True)
    )
    result["row_index"] = np.arange(
        len(result),
        dtype=np.int64,
    )
    return result


def run_year(year):
    samples_year = prepare_year_metadata(year)

    metadata_path = FINAL_OUTPUT_DIR / f"samples_nearest_valid_{year}.parquet"
    samples_year.to_parquet(metadata_path, index=False)

    zarr_path = FINAL_OUTPUT_DIR / f"s1_nearest_valid_{year}.zarr"
    root = create_year_zarr(zarr_path, len(samples_year))

    done = np.asarray(root["sample_done"][:], dtype=bool)

    cluster_ranges = (
        samples_year.groupby("cluster_id")["row_index"]
        .agg(start="min", end="max")
        .reset_index()
    )

    for cluster in tqdm(
        cluster_ranges.itertuples(index=False),
        total=len(cluster_ranges),
        desc=f"Nearest-valid aggregation {year}",
    ):
        cluster_id = int(cluster.cluster_id)
        row_start = int(cluster.start)
        row_end = int(cluster.end)
        row_slice = slice(row_start, row_end + 1)

        if done[row_slice].all():
            continue

        result = compile_cluster_year_from_shards(
            cluster_id,
            year,
            samples_year,
        )

        if result is None:
            data = np.full(
                (row_end - row_start + 1, N_OUTPUT_BANDS, PATCH_SIZE, PATCH_SIZE),
                np.nan,
                dtype=np.float32,
            )
            counts = np.zeros(
                (row_end - row_start + 1, N_REAL_BANDS),
                dtype=np.uint8,
            )
            offsets = np.full(
                (row_end - row_start + 1, 2),
                SOURCE_OFFSET_MISSING,
                dtype=np.int16,
            )
        else:
            data, counts, offsets = result

        # The cluster rows are contiguous because samples_year is sorted by cluster.
        if len(data) != (row_end - row_start + 1):
            raise RuntimeError(
                f"Cluster {cluster_id} has {len(data)} rows, but its Zarr slice expects "
                f"{row_end - row_start + 1}."
            )

        for index, band in enumerate(OUTPUT_BANDS):
            root[band][row_slice] = data[:, index]

        root["obs_count"][row_slice] = counts
        root["source_day_offset"][row_slice] = offsets

        # Set completion only after all feature arrays have been written.
        root["sample_done"][row_slice] = True
        done[row_slice] = True

    root.attrs["complete"] = bool(
        np.asarray(root["sample_done"][:], dtype=bool).all()
    )

    return metadata_path, zarr_path


## Running with live progress

Use **Kernel → Restart Kernel and Run All** (or run every definition cell in order) before the full-run cell so the updated timeout, cache helper, and heartbeat functions are active. The notebook keeps the current SQLite/cache paths so completed cluster checkpoints are reused. The STAC query bar shows individual requests; if one query is slow it remains visible as the active query with elapsed seconds.

**v6 heartbeat fix:** the cluster bar now reports cache/window/filter stages before the first STAC request; existing scene-backlog checks are bounded so they cannot block the main progress loop.


**v8 benchmark-first safety:** the final full-pipeline cell now defaults to OFF. With an existing SQLite inventory, run all definition cells and the COG benchmark; inspect its CSV outputs before choosing whether to change the production resampling method. If the inventory does not yet exist, set `RUN_BUILD_INVENTORY_ONLY = True` in the final cell and run it to build/resume metadata first. Only set `RUN_FULL_GLOBAL_PIPELINE = True` deliberately when ready for bulk extraction.


## Benchmark: exact 10 m → 1 km aggregation vs COG overview reads

**This is an evaluation step, not a production-method switch.** The production reader remains the existing native-resolution read + exact 100 × 100 pixel aggregation until you inspect the speed and error report.

The benchmark samples up to three scenes from early, middle, and late parts of the indexed acquisition timeline, checks that each has internal COG overviews and useful station patches, then compares both methods on the same station windows and polarizations:

- **Exact reference:** read the native 1300 × 1300 patch and use the notebook's current `pool_10m_to_1km` function.
- **Overview read:** read the same patch directly to 13 × 13 using Rasterio `Resampling.average`, allowing GDAL to select a suitable COG overview where available.

It records elapsed read/open time and numerical differences per patch: MAE, RMSE, 95th-percentile/max absolute error, bias, relative MAE, and nodata-mask agreement. It writes a CSV report under `OUTPUT_DIR`. SAS credentials and signed asset URLs are never written to that report.

The benchmark uses small representative patches, not a global run. Results depend on network/CDN cache state; use the per-patch results and aggregate results together. It **does not automatically approve** overview reads for production.

**Safety:** the global extraction cell below is disabled by default, so running the notebook to get the benchmark will not unexpectedly start the 204k-scene production job. Review the benchmark first; only enable the full pipeline deliberately.


In [22]:
# ============================================================
# COG OVERVIEW BENCHMARK -- EXACT REFERENCE VS DIRECT OVERVIEW READ
# ============================================================

from rasterio.enums import Resampling

RUN_COG_OVERVIEW_BENCHMARK = False
COG_BENCHMARK_SCENE_COUNT = 3
COG_BENCHMARK_STATIONS_PER_SCENE = 2
COG_BENCHMARK_CANDIDATES_PER_TIME_REGION = 36
COG_BENCHMARK_POLARIZATION_PAIRS = (("vv", "vh"),)  # VV/VH-only; never benchmark HH/HV


def _benchmark_pair_for_item(item):
    available = set(available_polarizations(item))
    for pair in COG_BENCHMARK_POLARIZATION_PAIRS:
        if set(pair).issubset(available):
            return tuple(pair)
    return None


def _benchmark_normalize_values(values, nodata):
    values = np.asarray(values, dtype=np.float32).copy()
    if nodata is not None:
        try:
            if not np.isnan(float(nodata)):
                values[values == np.float32(nodata)] = np.nan
        except (TypeError, ValueError):
            pass
    values[~np.isfinite(values)] = np.nan
    return values


def _benchmark_candidate_scene_rows(db_path, candidates_per_region=36):
    """Fetch small, evenly spaced slices from the day-sorted SQLite index."""
    if not Path(db_path).exists():
        print(f"Benchmark skipped: scene inventory database not found: {db_path}")
        return []

    conn = sqlite3.connect(db_path, timeout=60)
    try:
        count, min_day, max_day = conn.execute(
            "SELECT COUNT(*), MIN(day), MAX(day) FROM scenes"
        ).fetchone()
        count = int(count or 0)
        if count == 0:
            print("Benchmark skipped: scene inventory database is empty.")
            return []

        # Keep the time regions separate so the scene chooser must try one scene
        # from early, middle, and late acquisition periods before filling extras.
        quantiles = np.linspace(0.06, 0.94, int(COG_BENCHMARK_SCENE_COUNT))
        offsets = [int(round(float(q) * (count - 1))) for q in quantiles]
        per_region = max(8, int(candidates_per_region))
        regions = []
        seen = set()
        for offset in offsets:
            start = max(0, offset - per_region // 2)
            batch = conn.execute(
                "SELECT scene_id, day, item_json FROM scenes "
                "ORDER BY day, scene_id LIMIT ? OFFSET ?",
                (per_region, start),
            ).fetchall()
            unique_batch = []
            for row in batch:
                if row[0] not in seen:
                    seen.add(row[0])
                    unique_batch.append(row)
            regions.append(unique_batch)

        date_start = (pd.Timestamp("1970-01-01") + pd.Timedelta(days=int(min_day))).date()
        date_end = (pd.Timestamp("1970-01-01") + pd.Timedelta(days=int(max_day))).date()
        print(
            f"Benchmark candidate pool: {sum(map(len, regions))} rows sampled from "
            f"{count:,} scenes ({date_start} to {date_end}); "
            f"regions={[len(x) for x in regions]}"
        )
        return regions
    finally:
        conn.close()

def _choose_cog_benchmark_scenes(
    db_path=SCENE_INDEX_DB_PATH,
    scene_count=COG_BENCHMARK_SCENE_COUNT,
    stations_per_scene=COG_BENCHMARK_STATIONS_PER_SCENE,
    candidates_per_region=COG_BENCHMARK_CANDIDATES_PER_TIME_REGION,
):
    """Choose scenes spanning the indexed timeline, with overviews and eligible patches."""
    candidate_regions = _benchmark_candidate_scene_rows(
        db_path, candidates_per_region=candidates_per_region
    )
    if not candidate_regions:
        return []

    selected = []
    seen_ids = set()

    def evaluate_candidate(row):
        scene_id, scene_day, item_json = row
        if scene_id in seen_ids:
            return None
        seen_ids.add(scene_id)
        try:
            item = pystac.Item.from_dict(json.loads(item_json))
            pair = _benchmark_pair_for_item(item)
            if pair is None:
                return None

            station_candidates = _global_scene_station_candidates(item)
            needed = stations_needed_for_scene(item, station_candidates, SAMPLE_DAYS)
            if needed.empty:
                return None

            # The metadata open is intentionally outside timed measurements. It
            # verifies real internal overviews and finds fully covered 1300 px patches.
            href = rtc_token.sign(_clean_href(item.assets[pair[0]].href), force=False)
            with rasterio.Env(**GDAL_ENV):
                with rasterio.open(href) as src:
                    overviews = list(src.overviews(1))
                    if not overviews:
                        return None
                    pixels, pixel_valid = station_patch_windows(
                        src.transform, src.crs.to_string(), src.width, src.height, needed
                    )
                    fully_covered = (
                        pixel_valid
                        & (pixels[:, 0] >= 0)
                        & (pixels[:, 1] >= 0)
                        & (pixels[:, 2] <= src.width)
                        & (pixels[:, 3] <= src.height)
                        & ((pixels[:, 2] - pixels[:, 0]) == SOURCE_PATCH_PX)
                        & ((pixels[:, 3] - pixels[:, 1]) == SOURCE_PATCH_PX)
                    )
                    eligible = np.flatnonzero(fully_covered)
                    if len(eligible) == 0:
                        return None
                    take = min(int(stations_per_scene), len(eligible))
                    positions = np.linspace(0, len(eligible) - 1, take).round().astype(int)
                    chosen_positions = eligible[positions]
                    station_rows = []
                    for pos in chosen_positions:
                        row_data = needed.iloc[int(pos)]
                        c0, r0, c1, r1 = map(int, pixels[int(pos)])
                        station_rows.append({
                            "station_id": str(row_data["station_id"]),
                            "window": (c0, r0, c1, r1),
                        })

            result = {
                "item": item,
                "scene_id": str(item.id),
                "day": int(scene_day),
                "polarizations": pair,
                "overview_factors": overviews,
                "stations": station_rows,
            }
            print(
                f"Selected {item.id}: day={int(scene_day)}, pair={pair}, "
                f"overviews={overviews}, patches={len(station_rows)}"
            )
            return result
        except (
            OSError, ValueError, KeyError, TypeError,
            rasterio.errors.RasterioIOError, requests.RequestException,
        ) as exc:
            print(f"Skipping benchmark candidate {scene_id}: {type(exc).__name__}: {str(exc)[:140]}")
            return None

    # First aim for one representative scene from each temporal region.
    for region in candidate_regions:
        for row in region:
            candidate = evaluate_candidate(row)
            if candidate is not None:
                selected.append(candidate)
                break
        if len(selected) >= int(scene_count):
            break

    # If a region had no eligible scenes, fill from remaining sampled candidates.
    if len(selected) < int(scene_count):
        for region in candidate_regions:
            for row in region:
                candidate = evaluate_candidate(row)
                if candidate is not None:
                    selected.append(candidate)
                    if len(selected) >= int(scene_count):
                        break
            if len(selected) >= int(scene_count):
                break

    if len(selected) < int(scene_count):
        print(
            f"Warning: found {len(selected)} eligible scenes, requested {scene_count}. "
            "The benchmark will use those that qualified."
        )
    return selected

def _benchmark_read_patch(href, asset_key, window_tuple, strategy):
    """Time a fresh COG open plus one patch read. SAS-token time is excluded."""
    c0, r0, c1, r1 = map(int, window_tuple)
    window = Window(c0, r0, c1 - c0, r1 - r0)
    benchmark_env = dict(GDAL_ENV)
    # Disable per-handle VSI caching for both arms so one strategy doesn't
    # inherit its counterpart's local file-handle block cache.
    benchmark_env["VSI_CACHE"] = "FALSE"
    started = time.perf_counter()
    with rasterio.Env(**benchmark_env):
        with rasterio.open(href) as src:
            if strategy == "exact":
                native = read_block(src, window)
                result = pool_10m_to_1km(native)
            elif strategy == "overview":
                result = src.read(
                    1,
                    window=window,
                    out_shape=(PATCH_SIZE, PATCH_SIZE),
                    out_dtype="float32",
                    resampling=Resampling.average,
                )
                result = _benchmark_normalize_values(result, src.nodata)
            else:
                raise ValueError(f"Unknown benchmark strategy: {strategy}")
    return result, float(time.perf_counter() - started)


def _benchmark_patch_metrics(exact, overview):
    exact = np.asarray(exact, dtype=np.float32)
    overview = np.asarray(overview, dtype=np.float32)
    exact_valid = np.isfinite(exact)
    overview_valid = np.isfinite(overview)
    joint = exact_valid & overview_valid
    mask_agreement = float(np.mean(exact_valid == overview_valid))
    out = {
        "valid_exact": int(exact_valid.sum()),
        "valid_overview": int(overview_valid.sum()),
        "valid_joint": int(joint.sum()),
        "valid_mask_agreement_pct": 100.0 * mask_agreement,
        "mae": np.nan,
        "rmse": np.nan,
        "bias_overview_minus_exact": np.nan,
        "p95_abs_error": np.nan,
        "max_abs_error": np.nan,
        "mae_pct_of_mean_abs_exact": np.nan,
    }
    if joint.any():
        difference = overview[joint].astype(np.float64) - exact[joint].astype(np.float64)
        absolute = np.abs(difference)
        mae = float(np.mean(absolute))
        mean_abs_exact = float(np.mean(np.abs(exact[joint].astype(np.float64))))
        out.update({
            "mae": mae,
            "rmse": float(np.sqrt(np.mean(difference ** 2))),
            "bias_overview_minus_exact": float(np.mean(difference)),
            "p95_abs_error": float(np.percentile(absolute, 95)),
            "max_abs_error": float(np.max(absolute)),
            "mae_pct_of_mean_abs_exact": float(100.0 * mae / mean_abs_exact) if mean_abs_exact > 0 else np.nan,
        })
    return out


def run_cog_overview_benchmark(
    db_path=SCENE_INDEX_DB_PATH,
    scene_count=COG_BENCHMARK_SCENE_COUNT,
    stations_per_scene=COG_BENCHMARK_STATIONS_PER_SCENE,
):
    """Compare exact patch means with COG overview-based 13x13 reads on real scenes."""
    if not Path(db_path).exists():
        print(f"Benchmark skipped: scene inventory database not found: {db_path}")
        return pd.DataFrame(), pd.DataFrame()
    with sqlite3.connect(db_path, timeout=60) as _conn:
        _count = int(_conn.execute("SELECT COUNT(*) FROM scenes").fetchone()[0])
    if _count == 0:
        print("Benchmark skipped: scene inventory database contains no scenes.")
        return pd.DataFrame(), pd.DataFrame()
    chosen = _choose_cog_benchmark_scenes(
        db_path=db_path,
        scene_count=scene_count,
        stations_per_scene=stations_per_scene,
    )
    if not chosen:
        raise RuntimeError(
            "No eligible COG scenes with overviews and fully covered station patches were found. "
            f"PRODUCTION_READ_STRATEGY remains {PRODUCTION_READ_STRATEGY!r}."
        )

    rows = []
    task_index = 0
    for scene in chosen:
        item = scene["item"]
        for pol in scene["polarizations"]:
            base_href = _clean_href(item.assets[pol].href)
            # Make method-specific signed URLs before timing, reducing the chance
            # that one method reuses the other's GDAL URL/block-cache entries.
            exact_token = rtc_token.token(force=True)
            overview_token = rtc_token.token(force=True)
            exact_href = f"{base_href}?{exact_token}"
            overview_href = f"{base_href}?{overview_token}"

            for station in scene["stations"]:
                # Alternate order to counteract systematic warm/cold ordering.
                if task_index % 2 == 0:
                    order = (("exact", exact_href), ("overview", overview_href))
                else:
                    order = (("overview", overview_href), ("exact", exact_href))
                task_index += 1
                values = {}
                durations = {}
                for strategy, href in order:
                    array, seconds = _benchmark_read_patch(
                        href, pol, station["window"], strategy
                    )
                    values[strategy] = array
                    durations[strategy] = seconds

                metrics = _benchmark_patch_metrics(values["exact"], values["overview"])
                exact_seconds = durations["exact"]
                overview_seconds = durations["overview"]
                rows.append({
                    "scene_id": scene["scene_id"],
                    "scene_day": int(scene["day"]),
                    "station_id": station["station_id"],
                    "polarization": pol,
                    "overview_factors": ",".join(map(str, scene["overview_factors"])),
                    "exact_read_seconds": exact_seconds,
                    "overview_read_seconds": overview_seconds,
                    "speedup_exact_overview": exact_seconds / overview_seconds if overview_seconds > 0 else np.nan,
                    **metrics,
                })
                print(
                    f"{scene['scene_id']} | {station['station_id']} | {pol}: "
                    f"exact={exact_seconds:.2f}s, overview={overview_seconds:.2f}s, "
                    f"speedup={exact_seconds / overview_seconds if overview_seconds > 0 else np.nan:.2f}x, "
                    f"MAE={metrics['mae']:.6g}, RMSE={metrics['rmse']:.6g}, "
                    f"mask agreement={metrics['valid_mask_agreement_pct']:.1f}%"
                )

    results = pd.DataFrame(rows)
    if results.empty:
        raise RuntimeError("Benchmark produced no measurements; PRODUCTION_READ_STRATEGY is unchanged.")

    summary = results.groupby("polarization", dropna=False).agg(
        patch_tests=("scene_id", "count"),
        median_exact_seconds=("exact_read_seconds", "median"),
        median_overview_seconds=("overview_read_seconds", "median"),
        median_speedup=("speedup_exact_overview", "median"),
        median_mae=("mae", "median"),
        median_rmse=("rmse", "median"),
        worst_patch_p95_abs_error=("p95_abs_error", "max"),
        min_mask_agreement_pct=("valid_mask_agreement_pct", "min"),
        median_relative_mae_pct=("mae_pct_of_mean_abs_exact", "median"),
    ).reset_index()

    timestamp = pd.Timestamp.now(tz="UTC").strftime("%Y%m%dT%H%M%SZ")
    result_path = OUTPUT_DIR / f"cog_overview_benchmark_{timestamp}.csv"
    summary_path = OUTPUT_DIR / f"cog_overview_benchmark_summary_{timestamp}.csv"
    results.to_csv(result_path, index=False)
    summary.to_csv(summary_path, index=False)

    print("\n=== COG overview benchmark summary ===")
    print(summary.to_string(index=False))
    print("\nPer-patch results:", result_path)
    print("Summary:", summary_path)
    print(
        "\nThe benchmark does not change PRODUCTION_READ_STRATEGY. "
        f"Current production strategy: {PRODUCTION_READ_STRATEGY!r}. "
        "Overview mode is approximate; set the configuration to 'exact' if the "
        "measured numerical differences are not acceptable for the application."
    )
    return results, summary


if RUN_COG_OVERVIEW_BENCHMARK:
    cog_benchmark_results, cog_benchmark_summary = run_cog_overview_benchmark()
else:
    print("COG overview benchmark is disabled by default. Set RUN_COG_OVERVIEW_BENCHMARK=True to rerun it.")


COG overview benchmark is disabled by default. Set RUN_COG_OVERVIEW_BENCHMARK=True to rerun it.


In [23]:
# ============================================================
# OPTIONAL LATENCY DIAGNOSTICS (real network; writes nothing)
# ============================================================
# Set True once to see where the time goes and to confirm the optimisations.
RUN_LATENCY_DIAGNOSTICS = False

if RUN_LATENCY_DIAGNOSTICS:
    print("1) HTTP latency to the blob store (cold vs reused connection)")
    measure_http_latency()

    print("\n2) Direct reads vs the original boundless reads on real scenes")
    verify_direct_inside_reads()

    print("\n3) Per scene-band: seconds, ranged GETs and MB for both read paths")
    profile_scene_latency(n_scenes=2)
else:
    print("Latency diagnostics disabled. Set RUN_LATENCY_DIAGNOSTICS=True to run them.")


Latency diagnostics disabled. Set RUN_LATENCY_DIAGNOSTICS=True to run them.


## Optional: split scene extraction across multiple Python instances (no cloud spend)

This mode assigns every inventory scene to exactly one worker using a stable hash. Each worker reads only its assigned scenes and writes into a separate `_distributed_worker_XX_of_NN` folder. It does not run inventory searches or yearly aggregation. The same workflow works for multiple Python/Jupyter kernels on one laptop; separate VMs are optional, not required.

**First, stop any active scene-extraction run.** The scene inventory database must be complete and stable before workers start. With distributed mode disabled on the coordinator, run the helper function `export_distributed_worker_package()` once (execute cell 33 after setting `EXPORT_DISTRIBUTED_WORKER_PACKAGE = True`, or call the function directly). This verifies inventory completeness and creates `distributed_completed_scene_ids.txt`, which lets workers skip shards already completed by the coordinator. It also creates a ZIP for separate VMs; if all workers share the same laptop filesystem, you can use the existing files and do not need to unzip that package.

For a first test, use **two independent notebook copies/kernels** sharing the same project folder. In cell 1 of both copies, set `DISTRIBUTED_SCENE_WORKER_MODE = True` and `DISTRIBUTED_WORKER_COUNT = 2`. Set `DISTRIBUTED_WORKER_ID = 0` in the first and `1` in the second. Run all cells in both. The notebook automatically sets each process to 9 scene workers, keeping the combined target near the previously tested 18; don't use the same worker ID twice. Each worker uses its normal Planetary Computer token flow. Different tokens are not assumed to provide separate rate limits.

When both workers finish, on the coordinator set worker mode to False and run `merge_distributed_scene_shards()` once (or set `MERGE_DISTRIBUTED_SHARDS = True` and execute helper cell 33). This copies worker shards into the canonical v9 scene-shard folder. Then, on the coordinator, set `RUN_FULL_GLOBAL_PIPELINE = True` in the final cell and execute it: existing shards should be reused, and the remaining yearly aggregation runs centrally. Never run coordinator extraction or final Zarr aggregation while worker processes are still writing.

**For separate VMs:** use the ZIP plus a copy of this notebook on each VM, preserve the relative `Data/...` paths, and use the same worker count with unique IDs. Transfer the worker output folders back to the coordinator before merging.

This distributes legitimate work; it is not a way to disguise traffic or bypass service limits. Aggregate throughput may not scale linearly because the workers still share the same remote service, and local Python instances share the laptop's CPU, RAM, network adapter, and campus egress path.


In [24]:
# ============================================================
# DISTRIBUTED WORKER PACKAGE + SHARD MERGE HELPERS
# ============================================================
# Coordinator workflow:
#   1) Finish the SQLite inventory (all clusters complete) and stop the coordinator run.
#   2) Run export_distributed_worker_package() on the coordinator.
#   3) Copy/unzip the package and this notebook onto each worker's project root.
#   4) Set distributed mode True and a unique worker ID/count in cell 1.
# Worker workflow: Run all cells. The final cell performs ONLY that worker's scene reads.
# Afterward, copy each _distributed_worker_XX_of_NN folder back to the coordinator's
# SCENE_OBS_DIR and run merge_distributed_scene_shards() here with worker mode disabled.

import shutil
import zipfile


def _assert_scene_inventory_complete():
    expected_cluster_ids = set(clusters["cluster_id"].astype(int).tolist())
    if not SCENE_INDEX_DB_PATH.exists():
        raise FileNotFoundError(f"Scene inventory database not found: {SCENE_INDEX_DB_PATH}")
    with sqlite3.connect(SCENE_INDEX_DB_PATH, timeout=120) as conn:
        completed_cluster_ids = {
            int(row[0]) for row in conn.execute(
                "SELECT cluster_id FROM cluster_searches WHERE completed = 1"
            )
        }
        scene_count = int(conn.execute("SELECT COUNT(*) FROM scenes").fetchone()[0])
    missing = expected_cluster_ids - completed_cluster_ids
    unexpected = completed_cluster_ids - expected_cluster_ids
    if missing or unexpected:
        raise RuntimeError(
            "Scene inventory is not a consistent completed snapshot; do NOT distribute it. "
            f"Expected clusters={len(expected_cluster_ids)}, completed={len(completed_cluster_ids)}, "
            f"missing={len(missing)}, unexpected={len(unexpected)}. "
            "Run/resume the inventory on the coordinator until every cluster checkpoint is complete."
        )
    if scene_count == 0:
        raise RuntimeError("Scene inventory has zero scenes; refusing to create a worker package.")
    return len(expected_cluster_ids), scene_count


def export_distributed_worker_package():
    """Create a consistent DB snapshot, completed-shard manifest, and portable ZIP."""
    if DISTRIBUTED_SCENE_WORKER_MODE:
        raise RuntimeError("Export the worker package only from the coordinator (worker mode disabled).")
    expected_clusters, scene_count = _assert_scene_inventory_complete()
    package_dir = OUTPUT_DIR / "distributed_worker_package"
    package_dir.mkdir(parents=True, exist_ok=True)
    snapshot_db = package_dir / SCENE_INDEX_DB_PATH.name

    # SQLite backup API creates a consistent snapshot even if the source uses WAL.
    with sqlite3.connect(SCENE_INDEX_DB_PATH, timeout=120) as source_conn:
        with sqlite3.connect(snapshot_db) as target_conn:
            source_conn.backup(target_conn)

    existing_names = {p.name for p in SCENE_OBS_DIR.glob("*.npz") if p.is_file()}
    completed_ids = []
    with sqlite3.connect(snapshot_db) as conn:
        for (scene_id,) in conn.execute("SELECT scene_id FROM scenes ORDER BY scene_id"):
            if _scene_shard_path(str(scene_id)).name in existing_names:
                completed_ids.append(str(scene_id))

    manifest_path = OUTPUT_DIR / "distributed_completed_scene_ids.txt"
    manifest_path.write_text("".join(f"{sid}\n" for sid in completed_ids), encoding="utf-8")

    inputs = [
        Path("Data/Global/pipeline_stage1/stations.parquet"),
        Path("Data/Global/pipeline_stage1/station_date_candidates.parquet"),
    ]
    missing_inputs = [str(p) for p in inputs if not p.exists()]
    if missing_inputs:
        raise FileNotFoundError(
            "Cannot make a portable worker package because these required inputs are missing: "
            + ", ".join(missing_inputs)
        )

    package_meta = {
        "scene_inventory_db": SCENE_INDEX_DB_PATH.name,
        "scene_count": int(scene_count),
        "expected_clusters": int(expected_clusters),
        "coordinator_completed_shards": int(len(completed_ids)),
        "read_strategy": PRODUCTION_READ_STRATEGY,
        "remote_polarizations": list(READ_POLARIZATIONS),
        "nearest_days": int(MAX_NEAREST_DAYS),
        "created_utc": pd.Timestamp.now(tz="UTC").isoformat(),
        "note": "Each scene is assigned by stable SHA-256(scene_id) modulo worker count.",
    }
    meta_path = package_dir / "distributed_package_info.json"
    meta_path.write_text(json.dumps(package_meta, indent=2), encoding="utf-8")
    readme_path = package_dir / "README.txt"
    readme_path.write_text(
        "SENTINEL-1 DISTRIBUTED SCENE WORKER PACKAGE\n\n"
        "1. Extract this ZIP at the same project root used by the notebook, preserving the Data/ paths.\n"
        "2. Use the companion distributed-worker notebook. In cell 1 set:\n"
        "     DISTRIBUTED_SCENE_WORKER_MODE = True\n"
        "     DISTRIBUTED_WORKER_COUNT = N   (same N on every machine)\n"
        "     DISTRIBUTED_WORKER_ID = 0..N-1 (unique on every machine)\n"
        "3. Run all notebook cells. A worker only reads its assigned scenes and writes into its own subfolder.\n"
        "4. Do not run the inventory or yearly aggregation independently on the workers.\n"
        "5. When all workers finish, copy their _distributed_worker_XX_of_NN folders into the coordinator's\n"
        "   scene_observations_nearest_overview_v1 folder, then run merge_distributed_scene_shards() on the coordinator.\n"
        "6. Run the normal coordinator notebook with distributed mode disabled to finalize yearly Zarr outputs.\n\n"
        "Use each authorized worker's normal access credentials. Distributed assignment is for dividing work,\n"
        "not for evading service rate limits.\n",
        encoding="utf-8",
    )

    archive_path = OUTPUT_DIR / "distributed_worker_package.zip"
    root_relative_db = SCENE_INDEX_DB_PATH.as_posix()
    root_relative_manifest = manifest_path.as_posix()
    with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=3) as zf:
        zf.write(snapshot_db, arcname=root_relative_db)
        zf.write(manifest_path, arcname=root_relative_manifest)
        zf.write(meta_path, arcname=(package_dir / meta_path.name).as_posix())
        zf.write(readme_path, arcname=(package_dir / readme_path.name).as_posix())
        for path in inputs:
            zf.write(path, arcname=path.as_posix())
    print("Distributed package created:", archive_path)
    print("Inventory complete clusters:", f"{expected_clusters:,}")
    print("Unique scenes:", f"{scene_count:,}")
    print("Existing coordinator shards recorded in manifest:", f"{len(completed_ids):,}")
    print("Package includes the SQLite snapshot, completed-scene manifest, stations, and station-date inputs.")
    return archive_path


def merge_distributed_scene_shards():
    """Copy completed worker .npz shards into the coordinator's canonical scene shard folder."""
    if DISTRIBUTED_SCENE_WORKER_MODE:
        raise RuntimeError("Disable distributed worker mode on the coordinator before merging shards.")
    worker_dirs = sorted(
        p for p in SCENE_OBS_DIR.glob("_distributed_worker_*_of_*") if p.is_dir()
    )
    if not worker_dirs:
        print("No distributed worker directories found under:", SCENE_OBS_DIR)
        return {"copied": 0, "already_present": 0}
    copied = already_present = 0
    for worker_dir in worker_dirs:
        worker_shards = list(worker_dir.glob("*.npz"))
        print(f"Merging {len(worker_shards):,} shards from {worker_dir.name} ...")
        for src in worker_shards:
            dest = SCENE_OBS_DIR / src.name
            if dest.exists():
                already_present += 1
                continue
            shutil.copy2(src, dest)
            copied += 1
    print(f"Merge complete: copied={copied:,}; already present={already_present:,}.")
    print("Worker subfolders are kept as backups; they are not deleted.")
    return {"copied": copied, "already_present": already_present}


# These are manual actions. Keep both False during normal Run All.
EXPORT_DISTRIBUTED_WORKER_PACKAGE = False
MERGE_DISTRIBUTED_SHARDS = False
if EXPORT_DISTRIBUTED_WORKER_PACKAGE:
    export_distributed_worker_package()
if MERGE_DISTRIBUTED_SHARDS:
    merge_distributed_scene_shards()


In [ ]:
# ============================================================
# FULL GLOBAL RUN: INVENTORY -> UNIQUE SCENE READS -> YEARLY ZARR
# ============================================================

# In worker mode, this cell intentionally runs ONLY assigned scene reads.
# It never rebuilds the inventory and never writes yearly Zarr stores from workers.
RUN_BUILD_INVENTORY_ONLY = False
RUN_FULL_GLOBAL_PIPELINE = False

if DISTRIBUTED_SCENE_WORKER_MODE:
    print("DISTRIBUTED SCENE-WORKER MODE")
    print(f"Worker {DISTRIBUTED_WORKER_ID + 1}/{DISTRIBUTED_WORKER_COUNT}")
    print("No inventory searches or yearly Zarr finalization will run on this worker.")
    _assert_scene_inventory_complete()
    _ = _load_distributed_completed_ids()  # fail before work if the manifest is missing
    run_scene_extraction_from_db(
        db_path=SCENE_INDEX_DB_PATH,
        workers=SCENE_WORKERS,
    )
    print("Worker scene extraction finished.")
    print("Copy this worker's output folder back to the coordinator:", DISTRIBUTED_WORKER_OUTPUT_DIR)
    print("Do not run yearly aggregation here.")

elif RUN_BUILD_INVENTORY_ONLY:
    print("Building/resuming metadata inventory only; no raster scene extraction will run.")
    build_global_scene_index()

elif RUN_FULL_GLOBAL_PIPELINE:
    years = sorted(samples_c["year"].dropna().astype(int).unique())
    print("Years:", years[0], "->", years[-1])
    print("Total samples:", f"{len(samples_c):,}")
    print("Total stations:", f"{stations_cluster.station_id.nunique():,}")
    print("Total clusters:", clusters["cluster_id"].nunique())
    print("\n" + "=" * 70)
    print("1/3 BUILDING WINDOWED SQLITE INVENTORY (METADATA ONLY)")
    print("=" * 70)
    print("Nearest-date tolerance:", MAX_NEAREST_DAYS, "days")
    print("STAC timeout:", STAC_REQUEST_TIMEOUT_SECONDS, "seconds")
    print("Search workers:", SEARCH_WORKERS, "| Scene workers:", SCENE_WORKERS)
    print("Remote raster assets:", READ_POLARIZATIONS, "| HH/HV downloads: disabled")
    print("GDAL HTTP version preference:", GDAL_ENV.get("GDAL_HTTP_VERSION", "GDAL default"))
    print("Progress bars: cluster checkpoints + STAC requests; scene shards have a separate fail-fast stage")

    build_global_scene_index()
    with sqlite3.connect(SCENE_INDEX_DB_PATH) as conn:
        unique_scenes = conn.execute("SELECT COUNT(*) FROM scenes").fetchone()[0]
        associations = conn.execute("SELECT COUNT(*) FROM scene_clusters").fetchone()[0]
        completed_clusters = conn.execute(
            "SELECT COUNT(*) FROM cluster_searches WHERE completed=1"
        ).fetchone()[0]
    print("Unique scenes:", f"{unique_scenes:,}")
    print("Scene-cluster associations:", f"{associations:,}")
    print("Clusters committed:", f"{completed_clusters:,}/{len(clusters):,}")

    print("\n" + "=" * 70)
    print("2/3 VERIFYING / RETRYING ANY SCENE SHARDS NOT COMPLETED DURING INVENTORY")
    print("=" * 70)
    run_scene_extraction_from_db(db_path=SCENE_INDEX_DB_PATH, workers=SCENE_WORKERS)
    print("\nAll unique Sentinel-1 scene shards are checkpointed.")

    print("\n" + "=" * 70)
    print("3/3 FINAL NEAREST-VALID YEARLY AGGREGATION")
    print("=" * 70)
    for year in years:
        print(f"\nFinalizing {year}...")
        metadata_path, zarr_path = run_year(int(year))
        print("Metadata:", metadata_path)
        print("Zarr:", zarr_path)
    print("\nSentinel-1 RTC extraction complete.")
else:
    print("Global pipeline intentionally not started (safe default).")
    print("Production read strategy:", PRODUCTION_READ_STRATEGY)
    print("Scene shard directory:", SCENE_OBS_DIR)
    print("Set RUN_FULL_GLOBAL_PIPELINE = True only on the coordinator when ready")
    print("to verify shards and finalize the yearly Zarr outputs.")


DISTRIBUTED SCENE-WORKER MODE
Worker 1/2
No inventory searches or yearly Zarr finalization will run on this worker.
Planetary Computer SAS token acquired and cached before scene workers start.
Distributed worker 1/2; assigned scenes (including coordinator-completed IDs): 102,185
Worker shard directory: Data/Global/pipeline_stage1/sentinel1/scene_observations_nearest_overview_v1/_distributed_worker_00_of_02
Coordinator-completed scenes skipped: 21,655 total in manifest
Scene workers (start): 9 | band-read threads: 18
Direct inside-raster reads: OFF
Shard writer threads: 4
GDAL HTTP timeout: 30 seconds per HTTP operation
Back-off: halve concurrency after 3 consecutive failures (min 4 )


Scene shards completed:   0%|          | 0/102185 [00:00<?, ?scene/s]

Scene S1B_IW_GRDH_1SDV_20170530T175654_20170530T175719_005831_00A388_rtc failed (attempt 1/2): RasterioIOError: Read failed. See previous exception for details.. Retrying in 2s.
Scene S1B_IW_GRDH_1SDV_20170530T175654_20170530T175719_005831_00A388_rtc failed (1/3 consecutive): FileNotFoundError: [Errno 2] No such file or directory: 'Data/Global/pipeline_stage1/sentinel1/scene_observations_nearest_overview_v1/_distributed_worker_00_of_02/S1B_IW_GRDH_1SDV_20170530T175654_20170530T175719_005831_00A388_rtc_432df5edcf.npz.tmp' -> 'Data/Global/pipeline_stage1/sentinel1/scene_observations_nearest_overview_v1/_distributed_worker_00_of_02/S1B_IW_GRDH_1SDV_20170530T175654_20170530T175719_005831_00A388_rtc_432df5edcf.npz'
Remote read failed for S1B_IW_GRDH_1SDV_20170530T175629_20170530T175654_005831_00A388_rtc/vv (attempt 1/2): RasterioIOError: Read failed. See previous exception for details.. Retrying in 1s.
Scene S1B_IW_GRDH_1SDV_20170530T175629_20170530T175654_005831_00A388_rtc failed (attempt 

KeyboardInterrupt: 

Scene S1A_IW_GRDH_1SDV_20170531T223620_20170531T223645_016832_01BFBF_rtc failed (attempt 1/2): RasterioIOError: Read failed. See previous exception for details.. Retrying in 2s.
Remote read failed for S1B_IW_GRDH_1SDV_20170531T170038_20170531T170103_005845_00A3F0_rtc/vv (attempt 1/2): RasterioIOError: Read failed. See previous exception for details.. Retrying in 1s.
Scene S1B_IW_GRDH_1SDV_20170531T170038_20170531T170103_005845_00A3F0_rtc failed (attempt 1/2): RasterioIOError: Read failed. See previous exception for details.. Retrying in 2s.
Remote read failed for S1B_IW_GRDH_1SDV_20170531T170038_20170531T170103_005845_00A3F0_rtc/vv (attempt 1/2): RasterioIOError: Read failed. See previous exception for details.. Retrying in 1s.
Remote read failed for S1A_IW_GRDH_1SDV_20170601T001959_20170601T002024_016833_01BFC6_rtc/vv (attempt 1/2): RasterioIOError: Read failed. See previous exception for details.. Retrying in 1s.
Remote read failed for S1A_IW_GRDH_1SDV_20170601T133018_20170601T133043